# NIHONGA ? Phase 6: Architecture Healing

This phase follows PLAN.md: recover the original teacher's behavior by training the pruned student with matched prompt conditioning, latent and timestep. The pinned Qwen-Image 2.1 teacher has **32 blocks**, with hidden width **4096**; the plan's 60-block diagrams are illustrative. Phase 5 fitted four independent rank-256 bridge candidates. It did not select or assemble a healed student.

## 1. Verify and preserve the bridge-pretraining handoff

We verify the saved Phase 5 reports, checkpoint inspection, training states, selected weights, implementation files and all 30 teacher target bundles using their recorded checksums. We preserve the 20 TRAIN and 10 VALIDATION split and write a local manifest with four independent single-block student recipes. Each recipe replaces one original slot and keeps the other 31 transformer blocks; surviving hidden-state alignment uses original slot indices. No candidates are combined or selected as a winner.

The manifest binds the pinned teacher, scheduler, implementation, checkpoint and exact replay inputs. Teacher and student must receive identical inputs, and each student must rebuild its own prefix cache. Saved paired hidden targets cover candidate blocks only; supervision at surviving blocks requires additional capture. The plan's loss weights are recorded as suggestions, not a finalized training configuration.

This step uses only local files and standard Python. A completed rerun checks hashes and reads saved results, without repeating selection or tensor inspection. An incomplete handoff stops for inspection. No model forward, GPU job or optimizer update occurs in this preparation step.


In [1]:
# Persist the verified Phase 5 handoff without loading or running a model.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
if not (root / 'PLAN.md').exists():
    raise RuntimeError('Run from the NIHONGA project root.')
run_dir = root / 'data/phase6/handoff'
summary_path = run_dir / 'summary.json'
manifest_path = run_dir / 'manifest.json'

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

if summary_path.exists():
    summary = read_json(summary_path)
    require(summary['status'] == 'handoff_verified', 'Saved handoff is incomplete.')
    require(digest(manifest_path) == summary['manifest_sha256'], 'Saved manifest changed.')
    manifest = read_json(manifest_path)
    for name, expected in manifest['artifact_sha256'].items():
        require(digest(root / name) == expected, 'Handoff input changed: ' + name)
    print('Reusing verified handoff; no selection, tensor loading, model calls or training.')
else:
    require(not run_dir.exists(), 'Partial handoff exists; inspect it before retrying.')
    fit_dir = Path('data/phase5/bridge_fitting_pilot')
    fit = read_json(fit_dir / 'summary.json')
    inspection = read_json(fit_dir / 'local_checkpoint_inspection.json')
    capture = read_json('data/phase5/capture_full_summary.json')
    index = read_json('data/phase5/capture_full_index.json')
    source = read_json('data/phase5/capture_pilot_request.json')['request']
    require(fit['status'] == capture['status'] == inspection['status'] == 'passed', 'Phase 5 checks failed.')
    require(fit['request_key'] == inspection['request_key'], 'Checkpoint inspection provenance differs.')
    require(fit['total_optimizer_updates'] == 2000 and fit['validation_optimizer_updates'] == 0, 'Fitting split use differs.')
    require(source['num_teacher_blocks'] == 32 and source['hidden_dim'] == 4096, 'Teacher architecture differs.')
    require(digest('data/phase5/capture_full_index.json') == capture['index_sha256'], 'Target index changed.')
    fit_request = read_json(fit_dir / 'request.json')
    require(fit_request['request_key'] == fit['request_key'], 'Fitting request differs.')
    require(fit_request['request']['target_index'] == index, 'Fitting used a different target index.')
    train_ids = {r['source_id'] for r in index['train']}
    validation_ids = {r['source_id'] for r in index['validation']}
    require(len(train_ids) == 20 and len(validation_ids) == 10 and not train_ids & validation_ids, 'Pilot split differs.')
    paths = [
        'data/phase5/bridge_fitting_pilot/request.json',
        'data/phase5/bridge_fitting_pilot/summary.json',
        'data/phase5/bridge_fitting_pilot/local_checkpoint_inspection.json',
        'data/phase5/capture_pilot_request.json',
        'data/phase5/capture_pilot_rows.jsonl',
        'data/phase5/capture_full_index.json',
        'data/phase5/capture_full_summary.json',
        'data/phase4/bridge_definition.py',
        'data/phase4/bridge_block_definition.py',
        'data/phase4/transformer_qwenimage21_pinned.py',
    ]
    hashes = {name: digest(name) for name in paths}
    require(hashes['data/phase4/transformer_qwenimage21_pinned.py'] == source['transformer_source_sha256'], 'Pinned transformer source changed.')
    require(Path('data/phase4/bridge_definition.py').read_text(encoding='utf-8') == fit_request['request']['core_source'], 'Bridge definition differs from fitting.')
    for artifact in fit['artifacts']:
        name = (fit_dir / artifact['name']).as_posix()
        hashes[name] = digest(name)
        require(hashes[name] == artifact['sha256'], 'Fitting artifact changed: ' + name)
    for split in ('train', 'validation'):
        for record in index[split]:
            name = record['tensor_path']
            hashes[name] = digest(name)
            require(hashes[name] == record['tensor_sha256'], 'Teacher target changed: ' + name)
            hashes[record['report_path']] = digest(record['report_path'])
    require({r['layer'] for r in fit['results']} == {2, 3, 4, 5}, 'Candidate set differs.')
    candidates = []
    inspected = {r['layer']: r for r in inspection['selected']}
    for result in sorted(fit['results'], key=lambda r: r['layer']):
        layer = result['layer']
        require(inspected[layer]['step'] == result['selected_step'] and inspected[layer]['nonzero_up_projection'], 'Selected checkpoint inspection differs.')
        checkpoint = (fit_dir / result['selected_checkpoint']).as_posix()
        require(hashes[checkpoint] == result['selected_checkpoint_sha256'], 'Selected weights changed.')
        candidates.append({
            'candidate_id': f'single_bridge_block_{layer:02d}',
            'removed_teacher_slots': [layer], 'teacher_blocks': 32,
            'surviving_transformer_blocks': 31, 'bridge_count': 1,
            'original_slot_count': 32, 'checkpoint_path': checkpoint,
            'checkpoint_sha256': hashes[checkpoint], 'selected_update': result['selected_step'],
            'bridge_parameters': result['parameters'],
            'surviving_hidden_alignment': [{'teacher_slot': slot, 'student_slot': slot} for slot in range(32) if slot != layer],
            'cache_policy': 'discard all previous caches; rebuild student prefix using modified states; never reuse teacher KV caches',
        })
    manifest = {
        'version': 'phase6-handoff-v1', 'artifact_sha256': hashes,
        'phase5_fitting_request_key': fit['request_key'],
        'teacher': {k: source[k] for k in ('model_repo', 'model_revision', 'config_sha256', 'num_teacher_blocks', 'hidden_dim', 'dtype', 'scheduler_class', 'scheduler_config', 'num_inference_steps', 'true_cfg_scale', 'runtime')},
        'candidates': candidates, 'target_index': index,
        'candidate_policy': 'independent single-block students; no joint insertion and no winner selected',
        'input_policy': 'teacher and student receive identical saved prompt conditioning, latent and measured timestep',
        'data_policy': 'TRAIN only for gradients; VALIDATION for diagnostics/selection; no internal test or benchmark optimization',
        'readiness': 'handoff only; assembled BF16 student replay and gradient/cache compatibility are untested',
        'healing_losses_from_plan': {'output_weight': 1.0, 'hidden_weight': 0.1, 'optional_bridge_weight': 0.1, 'status': 'starting suggestions; no training configuration selected'},
        'hidden_target_limit': 'cached paired states cover candidate blocks only; surviving-block hidden supervision needs additional capture',
    }
    run_dir.mkdir(parents=True)
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
    summary = {
        'status': 'handoff_verified', 'manifest_sha256': digest(manifest_path),
        'candidate_count': len(candidates), 'train_prompts': len(train_ids),
        'validation_prompts': len(validation_ids), 'saved_stage_inputs': 90,
        'verified_artifacts': len(hashes), 'teacher_calls': 0, 'student_calls': 0,
        'optimizer_updates': 0, 'winner_selected': False,
    }
    with summary_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(summary, indent=2) + '\n')
    print('Phase 6 handoff saved; all referenced artifacts verified.')
print('Independent candidates:', summary['candidate_count'])
print('Saved prompts: TRAIN', summary['train_prompts'], '| VALIDATION', summary['validation_prompts'])
print('Saved stage inputs:', summary['saved_stage_inputs'], '| verified artifacts:', summary['verified_artifacts'])
print('Teacher calls:', summary['teacher_calls'], '| student calls:', summary['student_calls'], '| optimizer updates:', summary['optimizer_updates'])
print('Manifest:', manifest_path.relative_to(root))


Phase 6 handoff saved; all referenced artifacts verified.
Independent candidates: 4
Saved prompts: TRAIN 20 | VALIDATION 10
Saved stage inputs: 90 | verified artifacts: 107
Teacher calls: 0 | student calls: 0 | optimizer updates: 0
Manifest: data\phase6\handoff\manifest.json


## What the handoff results mean

**The Phase 6 inputs are preserved and verified.** We recorded four independent candidate recipes, for blocks 2, 3, 4 and 5, and verified **107 referenced artifacts** against the available source checksums. That count includes training checkpoints, reports, implementation files and teacher bundles; it measures artifact coverage, not model quality. Checkpoint shape and finiteness checks reuse the saved passing Phase 5 inspection. No tensor was loaded again.

Each candidate retains **31 transformer blocks plus one bridge in 32 original slots**. This describes four alternative students, not a student with four removed blocks. The checkpoint updates selected by Phase 5 validation are 500 for block 2, 300 for block 3, 400 for block 4 and 300 for block 5. These are existing checkpoint choices; this step performs no new model selection.

The preserved dataset contains **20 TRAIN prompts and 10 VALIDATION prompts**, with disjoint source IDs. Each prompt has early, middle and late saved teacher inputs, giving **90 stage inputs**. Inputs include prompt conditioning, latent and measured timestep; the original target index locates the corresponding bundles and their teacher velocities. These are three snapshots of each teacher trajectory, not 90 independent prompts. Validation already participated in Phase 5 checkpoint selection, so it is not an independent final test. This remains a small implementation pilot and does not establish broad dataset coverage.

**Zero teacher calls, zero student calls and zero optimizer updates** occurred. No healed model or image-quality measurement exists yet. The verified handoff supports proceeding to an assembled-student compatibility measurement while preserving every candidate. It does not establish that any candidate is safe to accept: BF16 insertion behavior, downstream velocity changes, prefix-cache effects and gradient compatibility remain uncertain. Teacher prefix caches must never be reused in the student because downstream states change after insertion.

For hidden-state distillation, original slot identity gives the alignment for surviving blocks. Existing Phase 5 pairs provide local targets at the candidate blocks; they do not supply the surviving-block targets required by the full hidden loss. The suggested loss weights 1.0, 0.1 and 0.1 remain starting hypotheses. No optimizer, training budget or healing dataset has been committed by this manifest.

The local files are `data/phase6/handoff/manifest.json` and `summary.json`. They reference the preserved Phase 4/5 artifacts, which remain unchanged. This new handoff is local; no remote upload occurred. Completed reruns verify referenced hashes and read the saved summary without model imports, network calls or recomputing diagnostics.


## 2. Compare independent assembled students on saved validation inputs

We load the frozen pinned teacher once on the existing Modal H100 runtime and replay the 10 saved VALIDATION prompts at stages 0, 20 and 39. Each intact-teacher control must match its saved velocity within relative L2 tolerance 0.001. Then we independently replace block 2, 3, 4 or 5 with its bridge. For each candidate we compare the identity baseline and selected trained weights in the assembled BF16 model. Surviving blocks remain frozen and unchanged; no bridges are combined.

Every teacher/candidate/variant/prompt receives a fresh prefix cache, populated by the saved extraction call. Middle and late calls use that run's rebuilt prefix and exactly the saved teacher latent, conditioning and timestep. No teacher cache is shared with a student. Identity uses the selected down projection with its up projection zeroed, so it performs no residual correction. The trained variant uses the complete selected checkpoint.

We measure full image-token velocity differences in FP32, preserving per-prompt, per-stage metrics and every velocity tensor remotely before downloading checksummed results. We report unweighted means over prompts and counts of improvements against identity. There are 30 teacher control forwards and 240 student forwards: 10 prompts ? 3 stages ? 4 candidates ? 2 variants. No trajectory, image, gradient or optimizer update is run. Cached validation already selected Phase 5 checkpoints and is not an independent final test.

The immutable request binds the handoff, implementation and checkpoints. Completed reruns verify checksums and read results without model or network imports; incomplete remote attempts stop for inspection rather than repeating unsaved forwards. This is an inference compatibility check before healing, not an acceptance benchmark or a candidate selection step.


In [2]:
# Compare independent BF16 students on cached validation inputs, with durable reuse.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/student_replay_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-independent-student-replay-v1',
    'handoff_manifest_sha256': digest(handoff_path),
    'teacher': handoff['teacher'], 'candidates': handoff['candidates'],
    'validation': handoff['target_index']['validation'],
    'fit_key': handoff['phase5_fitting_request_key'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'core_source': Path('data/phase4/bridge_definition.py').read_text(encoding='utf-8'),
    'wrapper_source': Path('data/phase4/bridge_block_definition.py').read_text(encoding='utf-8'),
    'worker_source': worker_source,
    'variants': ['identity', 'trained'], 'stages': [0, 20, 39],
    'teacher_control_relative_l2_tolerance': 0.001,
    'precision': 'BF16 pipeline weights and residual addition; FP32 metrics',
    'cache_policy': 'fresh prefix cache for every teacher/candidate/variant/prompt; rebuild at extraction',
    'metrics': 'full target-image velocity relative L2, MSE, RMSE, maximum absolute difference and cosine; unweighted prompt means by stage',
    'selection': 'no winner selected; retain all per-prompt outputs and regressions',
    'scope': 'held-out fixed-input inference compatibility; no optimizer, gradient test, images or student trajectories',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/student_replay')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/replay/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Replay report provenance differs.')
    if report['status'] != 'passed':
        raise RuntimeError('Replay failed; inspect saved attempt, no automatic retry: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 30 or report['student_calls'] != 240 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected replay counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-student-replay')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env({'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume})(namespace['replay_students'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Student replay:', report['status'], '| GPU:', report['gpu'])
print('Teacher control calls:', report['teacher_calls'], '| student calls:', report['student_calls'], '| updates:', report['optimizer_updates'])
for row in report['aggregates']:
    print(row['candidate_id'], '|', row['stage'], '| identity L2:', round(row['identity_mean_relative_l2'], 6),
          '| trained L2:', round(row['trained_mean_relative_l2'], 6), '| improved prompts:', str(row['improved_prompts']) + '/10')
print('Saved report and velocity tensors:', run_dir)


RuntimeError: Replay failed; inspect saved attempt, no automatic retry: RuntimeError: Intact teacher control exceeds 0.001 relative L2.

## What the replay attempt tells us

**The intact-teacher control failed before any bridge was evaluated.** The pinned model loaded on an NVIDIA H100 80GB GPU and replayed the early-stage input for validation prompt `p1-4a84d737d801`. It was compared with the exact teacher velocity saved during Phase 5 capture.

| Measurement | Observed value | Meaning |
|---|---:|---|
| Relative L2 velocity difference | 0.00401626 = **0.4016%** | Norm of prediction difference divided by the saved teacher prediction norm |
| Allowed relative L2 difference | 0.001 = **0.1%** | Saved control tolerance |
| Velocity MSE | 0.0000716419 | Mean squared difference over all predicted image-token components |
| Velocity RMSE | 0.00846415 | Square root of MSE, in the model's velocity units |
| Maximum absolute velocity difference | 0.109375 | Largest component difference; it is not a percentage |
| Flattened velocity cosine | 0.99999273 | Strong agreement in direction, which does not establish replay equivalence |

The relative difference is approximately **4.02 times the tolerance**. Although cosine is near one, the numerical discrepancy exceeds the fixed replay gate. MSE and RMSE are model-output errors, not image-quality scores. We kept the tolerance unchanged and did not accept this control as passing.

The attempt executed **one intact-teacher forward, zero student forwards and zero optimizer updates**. The middle and late controls, all four identity students and all four trained students were not evaluated. There are no candidate rankings, velocity-recovery measurements, images or healed checkpoints from this step. The previously passing Phase 5 controls were run in the original capture process; this new cross-process replay shows that replay equivalence has not yet been established in this runtime.

The model source, configuration, scheduler, selected bridge checkpoints and first target bundle passed their provenance checks before this forward. Those checks narrow the investigation but do not prove identical numerical execution. The report does not establish the cause of the mismatch. Attention execution, precision/runtime settings and reconstruction of the saved call remain hypotheses requiring evidence; the discrepancy must not be attributed to a bridge because none was inserted.

**Decision supported:** pause candidate comparisons until the intact teacher can reproduce the saved target under a justified replay contract. **Still uncertain:** why the loaded teacher differs, whether the same discrepancy affects other stages/prompts, and whether any trained bridge improves complete-model behavior.

The immutable local request and failed report are `data/phase6/student_replay/request.json` and `summary.json`; the remote attempt also preserves its started marker, request and report. Execution logs are saved locally. The failed report retains the control metrics; no `velocities.pt` artifact was produced because the failure occurred before the successful-run tensor commit. No previous result was overwritten.

A rerun was checked: it reads the saved failure and raises the same error without importing Modal, Torch or Diffusers, contacting the network or repeating any forward. Handoff and replay JSON artifacts remained unchanged. The error output above is intentional evidence of the failed control, not a completed student evaluation.


## 3. Diagnose fresh-teacher replay and intermediate-state repeatability

The prior attempt stopped because the intact teacher differed from its saved early-stage output. We preserve that failed request and report. This independent diagnostic uses only its first VALIDATION prompt and the existing teacher bundle. It does not rerun the original 40-step trajectory or recapture training data.

We load the pinned teacher on an H100 and replay the three saved inputs twice, rebuilding the prefix cache for each run. This costs six intact-teacher forwards. We compare each velocity with the saved capture, and compare the two runs with each other. Forward hooks also compare sampled inputs and outputs of blocks 2, 3, 4 and 5 with their saved paired states. That can reveal whether a discrepancy is already present before a candidate block or arises later.

We record package versions, CUDA and numerical settings, attention processor/backend, input tensor layout, and small model buffers. The timestep-frequency buffer is compared with the canonical formula in the pinned source without modifying it. The runtime pipeline source is saved for inspection. No alternative precision, attention backend, weights or tolerance is imposed, and no bridge is inserted.

We persist velocities, sampled hidden tensors, buffers and diagnostics remotely even if the original replay tolerance still fails, then download them with checksum verification. A completed diagnostic is not a passed teacher control. Reruns reuse its saved measurements without forwards or network imports; partial attempts stop for inspection. This step investigates one prompt and cannot establish equivalence across all validation inputs.


In [3]:
# Diagnose the saved teacher-control failure without inserting a bridge.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/teacher_diagnostic_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
failed_path = Path('data/phase6/student_replay/summary.json')
failed = read_json(failed_path)
if failed['status'] != 'failed' or failed['student_calls'] != 0:
    raise RuntimeError('Diagnostic requires the preserved teacher-only failed attempt.')
request = {
    'version': 'phase6-teacher-replay-diagnostic-v1',
    'handoff_manifest_sha256': digest(handoff_path),
    'failed_report_sha256': digest(failed_path),
    'teacher': handoff['teacher'],
    'record': handoff['target_index']['validation'][0],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'runs': ['fresh_1', 'fresh_2'], 'stages': [0, 20, 39],
    'control_relative_l2_tolerance': 0.001,
    'policy': 'two fresh-cache intact-teacher replays; sampled paired states compared with capture; no bridge, changed precision, trajectory or optimizer',
    'persistence': 'save measured velocities, sampled hidden states, runtime buffers, pipeline source and diagnostics even when control tolerance fails',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/teacher_diagnostic')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/teacher_diagnostic/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Replay report provenance differs.')
    if report['status'] != 'diagnostic_complete':
        raise RuntimeError('Replay failed; inspect saved attempt, no automatic retry: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 6 or report['student_calls'] != 0 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected replay counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-teacher-diagnostic')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env({'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['diagnose_teacher'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Teacher diagnostic:', report['status'], '| GPU:', report['gpu'], '| original control passed:', report['control_passed'])
print('Teacher calls:', report['teacher_calls'], '| student calls:', report['student_calls'], '| updates:', report['optimizer_updates'])
for row in report['velocity_metrics']:
    print(row['run'], row['stage'], '| saved-target relative L2:', round(row['relative_l2_error'], 8))
for row in report['repeatability']:
    print('Repeatability', row['stage'], '| relative L2:', round(row['relative_l2_error'], 8), '| exactly equal:', row['exact_equal'])
print('Temporal frequency buffer versus canonical values:', report['temporal_frequency_comparison'])
print('Reports, measured tensors and runtime pipeline source:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Teacher diagnostic: diagnostic_complete | GPU: NVIDIA H100 80GB HBM3 | original control passed: False
Teacher calls: 6 | student calls: 0 | updates: 0
fresh_1 early | saved-target relative L2: 0.00419176
fresh_1 middle | saved-target relative L2: 0.00314055
fresh_1 late | saved-target relative L2: 0.02385053
fresh_2 early | saved-target relative L2: 0.00419176
fresh_2 middle | saved-target relative L2: 0.00314055
fresh_2 late | saved-target relative L2: 0.02385053
Repeatability early | relative L2: 0.0 | exactly equal: True
Repeatability middle | relative L2: 0.0 | exactly equal: True
Repeatability late | relative L2: 0.0 | exactly equal: True
Temporal frequency buffer versus canonical values: {'relative_l2_error': 0.0, 'max_abs_error': 0.0, 'rmse': 0.0, 'exact_equal': True}
Reports, measured tensors and runtime pipeline source: data\phase6\teacher_diagnostic


## What the teacher diagnostic results mean

**The fresh teacher is repeatable within this loaded process, but it still does not reproduce the saved Phase 5 targets within tolerance.** We completed two fresh-cache replays of the same three saved inputs for validation prompt `p1-4a84d737d801`. This diagnostic used the intact teacher only: **six teacher forwards, zero student forwards and zero optimizer updates**. No training data was recaptured and no trajectory or image was generated.

### Differences from the saved teacher velocities

Both runs produced the same measurements. Relative L2 divides the prediction-difference norm by the saved target norm; RMSE and maximum absolute difference are in model velocity units, not percentages or image-quality scores.

| Stage | Relative L2 difference | Velocity RMSE | Maximum absolute difference |
|---|---:|---:|---:|
| early | 0.4192% | 0.008834 | 0.203125 |
| middle | 0.3141% | 0.005243 | 0.031250 |
| late | 2.3851% | 0.035632 | 0.171875 |

The fixed tolerance is **0.1%**. The observed differences exceed it by approximately **4.19 times at early**, **3.14 times at middle** and **23.85 times at late**. The late discrepancy is materially larger than the first failed early control. This does not establish a visible image-quality difference because no images or student trajectories were tested.

### Repeatability and where hidden states start to differ

Comparing run 2 with run 1 gives **zero relative L2 difference and exactly equal velocity tensors at all three stages**. Each run rebuilt its own prefix cache. This argues against ordinary run-to-run variation as the explanation within this particular process. Two repetitions of one prompt do not prove repeatability across processes, GPU allocations, prompts or other numerical settings. The earlier failed attempt's early difference was 0.4016%, while this independent diagnostic measured 0.4192%; the diagnostic does not isolate the reason for that cross-attempt difference.

Forward hooks compared exactly the sampled token positions saved in Phase 5. These measurements combine the sampled token roles present at each stage; they do not describe every token in the model.

| Stage | Sampled input to block 2: difference from capture | Sampled output of block 5: difference from capture |
|---|---:|---:|
| early | 0.0221% | 0.0752% |
| middle | 0.0309% | 0.2182% |
| late | 0.0487% | 0.2518% |

A nonzero difference is already present **before block 2**, the earliest candidate block. On the sampled states, differences are larger by the output of block 5. This narrows the issue to the intact teacher's computation upstream of and through these blocks; no learned bridge can be responsible because none was inserted. It does not identify the precise first differing operation, and sampled hidden-state errors should not be compared numerically with full-output velocity errors as if they were the same metric.

### Runtime evidence

Recorded package versions match the original capture: Torch **2.14.0**, Diffusers **0.41.0.dev0**, Transformers **5.17.0**, Accelerate **1.15.0** and Pillow **12.3.0**. Model source, model configuration and target-bundle checksums passed. Version agreement does not guarantee identical kernels or all runtime settings.

The teacher's only registered buffer reported here is the **128-value FP32 timestep-frequency buffer**. It is finite and **exactly matches** the canonical formula in the pinned source, with zero relative and maximum absolute difference. These results provide no evidence that this frequency buffer is corrupted. Its original capture-time tensor and all original numerical runtime settings were not separately recorded, so this check alone cannot prove complete runtime equivalence.

The fresh process used `QwenImage21AttnProcessor` with its backend selector unset, FP32 matmul precision `highest`, TF32 disabled and BF16 reduced-precision reduction enabled. Flash, memory-efficient and math SDPA were enabled; this records available settings and does not prove which attention kernel executed. Deterministic algorithms were not forced. These settings were observed without changing them. The runtime pipeline source is saved for inspection.

**Decision supported:** keep the original teacher-control gate and defer student comparisons. The teacher can give a stable but different answer from the saved capture, and the difference exists before any candidate replacement. **Still uncertain:** the first differing operation, why runtime/call reconstruction changes the result across capture and fresh loading, whether this affects the other nine validation prompts, and any bridge's complete-model behavior. No root cause or correction has been established by this diagnostic.

The immutable request, complete report, `diagnostic_tensors.pt` and `pipeline_qwenimage21_runtime.py` are saved under `data/phase6/teacher_diagnostic/` and backed up remotely. The tensor artifact contains all six measured velocities, all 48 sampled input/output tensors and the small-buffer snapshots, including the canonical frequency reference. The downloaded tensor bundle occupies approximately **101.02 MiB**. Original Phase 4/5 artifacts and the failed Phase 6 attempt remain unchanged.

The diagnostic cell was rerun using these saved files: no Torch, Diffusers or Modal import, network call, model forward, tensor inspection or diagnostic calculation occurred, and all saved diagnostic files and failed-attempt JSON files remained unchanged. The prior notebook error remains the preserved failed control; this cell's `diagnostic_complete` status means the investigation ran, not that the original control passed.


## 4. Test the original capture environment

Source inspection found a concrete runtime discrepancy: Phase 5 teacher capture set only `HF_HUB_OFFLINE=1` and `TRANSFORMERS_OFFLINE=1`, whereas the Phase 6 replay and diagnostic additionally set `CUBLAS_WORKSPACE_CONFIG=:4096:8`. That environment setting was carried over from bridge fitting. It is a hypothesis for the numerical mismatch, not yet a confirmed cause.

We create a separate immutable verification attempt using the original capture environment, pinned teacher, unchanged saved inputs and original 0.001 relative L2 tolerance. We test the first failing validation prompt first. If all its stages pass, we continue through all 20 TRAIN and 10 VALIDATION cached prompts, for up to 90 intact-teacher controls. Each prompt rebuilds its own prefix cache. The worker stops at the first failed stage.

Every measured velocity and its diagnostics are committed remotely per prompt and downloaded with checksum verification, including a failed control. No original captures, bridge weights, precision settings or tolerances are replaced. No student is run or trained. A passing result must cover all 90 controls. Completed reruns read saved results without forwards, model or network imports; incomplete attempts stop for inspection.


In [4]:
# Test the original capture environment and verify saved teacher controls.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/teacher_runtime_verification_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-original-capture-runtime-verification-v1',
    'handoff_manifest_sha256': digest(handoff_path),
    'teacher': handoff['teacher'],
    'records': [handoff['target_index']['validation'][0]] + handoff['target_index']['train'] + handoff['target_index']['validation'][1:],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'control_relative_l2_tolerance': 0.001,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'},
    'hypothesis': 'Phase 6 injected CUBLAS_WORKSPACE_CONFIG=:4096:8 absent from Phase 5 capture; restore capture environment without altering weights, inputs or tolerance',
    'policy': 'verify first failing prompt then all 30 cached prompts if controls pass; fresh teacher prefix cache per prompt; stop at first failed stage; persist each measured output',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/runtime_verification')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/runtime_verification/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Verification provenance differs.')
    if report['status'] not in ('passed', 'control_mismatch'):
        raise RuntimeError('Runtime verification failed; preserve attempt: ' + report.get('error', 'unknown'))
    if report['student_calls'] != 0 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected student or optimizer use.')
    if report['status'] == 'passed' and report['teacher_calls'] != 90:
        raise RuntimeError('Passing verification did not cover all 90 controls.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-original-runtime-verification')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['verify_teacher_runtime'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Original-runtime verification:', report['status'], '| GPU:', report['gpu'])
print('CUBLAS_WORKSPACE_CONFIG:', report['cublas_workspace_config'])
print('Teacher calls:', report['teacher_calls'], '| students:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Completed prompts:', report['completed_prompts'], '| exactly equal controls:', report['exact_equal_controls'])
print('Maximum relative L2 difference:', report['max_relative_l2_error'])
print('Saved report and outputs:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Original-runtime verification: control_mismatch | GPU: NVIDIA H100 80GB HBM3
CUBLAS_WORKSPACE_CONFIG: None
Teacher calls: 1 | students: 0 | updates: 0
Completed prompts: 1 | exactly equal controls: 0
Maximum relative L2 difference: 0.0040162187069654465
Saved report and outputs: data\phase6\runtime_verification


## What the original-environment test means

**Removing the extra workspace setting did not restore replay equivalence.** With `CUBLAS_WORKSPACE_CONFIG` absent, the first validation prompt's early teacher prediction still differed from its saved target by **0.4016219%**, against the unchanged **0.1%** tolerance. Its maximum absolute component difference was **0.109375** in velocity units. The previous unhooked attempt with the setting present measured 0.4016262%, essentially the same relative error. This test does not support that setting as the solution to the observed discrepancy.

The worker stopped after **one teacher forward**. It did not evaluate the remaining stages or prompts because the first control failed; it ran **zero students and zero optimizer updates**. `control_mismatch` is a completed negative result, not a passed verification. Model inputs, weights, precision and tolerance stayed unchanged. The full prediction tensor and diagnostics were preserved locally and remotely under `data/phase6/runtime_verification/`.

**Decision supported:** investigate another explanation instead of attributing the discrepancy to this environment setting or loosening the tolerance. **Still uncertain:** the first differing operation, role-specific hidden-state behavior, any relevant difference introduced during the original pipeline's use, and whether a reconstructed teacher can match all saved stages. The earlier failed attempts remain unchanged. The rerun of this cell reads the saved negative result without new forwards or network calls.


## 5. Separate saved hidden-state differences by token role

We analyze the tensors already saved by the fresh-teacher diagnostic, without another model call. Each sampled input/output is compared with its Phase 5 counterpart at the same block, stage and token position. Prefix text and image tokens are reported separately because prefix uses zero-timestep conditioning, while image tokens use the sampled diffusion timestep. We save relative L2 differences and the number of changed components. This may narrow the discrepancy to a token role, but sampled states cannot identify an exact upstream operation by themselves. Reruns verify source checksums and read the saved analysis without loading tensors or recalculating metrics.


In [5]:
# Analyze saved sampled differences by token role; no GPU calls.
import hashlib
import json
from pathlib import Path

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

report_path = Path('data/phase6/teacher_diagnostic/summary.json')
request = json.loads(Path('data/phase6/teacher_diagnostic/request.json').read_text())['request']
report = json.loads(report_path.read_text())
bundle_path = Path(request['record']['tensor_path'])
diagnostic_path = Path('data/phase6/teacher_diagnostic/diagnostic_tensors.pt')
expected = next(a['sha256'] for a in report['artifacts'] if a['name'] == diagnostic_path.name)
if digest(bundle_path) != request['record']['tensor_sha256'] or digest(diagnostic_path) != expected:
    raise RuntimeError('Saved diagnostic tensors differ.')
path = Path('data/phase6/role_difference_analysis.json')
source_hashes = {'bundle': digest(bundle_path), 'diagnostic': expected, 'report': digest(report_path)}
if path.exists():
    analysis = json.loads(path.read_text())
    if analysis['source_sha256'] != source_hashes:
        raise RuntimeError('Role analysis provenance differs.')
    print('Reusing saved role analysis; no tensor loading or metric calculation.')
else:
    import torch
    bundle = torch.load(bundle_path, weights_only=True, map_location='cpu')
    diagnostic = torch.load(diagnostic_path, weights_only=True, map_location='cpu')
    saved = {(p['stage'], p['layer']): p for p in bundle['pairs']}
    rows = []
    for observed in diagnostic['sampled_pairs']:
        if observed['run'] != 'fresh_1':
            continue
        pair = saved[(observed['stage'], observed['layer'])]
        target = pair['input' if observed['role'] == 'input' else 'target']
        for role_id, role in ((0, 'prefix'), (1, 'image')):
            mask = pair['token_roles'] == role_id
            if not mask.any():
                continue
            x = observed['value'][:, mask].float()
            y = target[:, mask].float()
            d = x - y
            rows.append({'stage': observed['stage'], 'layer': observed['layer'], 'state': observed['role'],
                'role': role, 'tokens': int(mask.sum()), 'relative_l2_error': float(d.norm()/y.norm().clamp_min(1e-12)),
                'changed_components': int((d != 0).sum()), 'components': d.numel(),
                'max_abs_error': float(d.abs().max()), 'exact_equal': bool(torch.equal(x, y))})
    analysis = {'source_sha256': source_hashes, 'status': 'complete', 'rows': rows,
                'teacher_calls': 0, 'student_calls': 0, 'optimizer_updates': 0}
    with path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(analysis, indent=2) + '\n')
for row in analysis['rows']:
    print(row['stage'], 'block', row['layer'], row['state'], row['role'], '| relative L2:', round(row['relative_l2_error'], 8),
          '| changed components:', str(row['changed_components']) + '/' + str(row['components']))
print('Saved role analysis:', path)


Reusing saved role analysis; no tensor loading or metric calculation.
early block 2 input prefix | relative L2: 0.0 | changed components: 0/65536
early block 2 input image | relative L2: 0.00062572 | changed components: 390785/1048576
early block 2 output prefix | relative L2: 0.0 | changed components: 0/65536
early block 2 output image | relative L2: 0.00106761 | changed components: 555237/1048576
early block 3 input prefix | relative L2: 0.0 | changed components: 0/65536
early block 3 input image | relative L2: 0.00106761 | changed components: 555237/1048576
early block 3 output prefix | relative L2: 0.0 | changed components: 0/65536
early block 3 output image | relative L2: 0.00162398 | changed components: 645331/1048576
early block 4 input prefix | relative L2: 0.0 | changed components: 0/65536
early block 4 input image | relative L2: 0.00162398 | changed components: 645331/1048576
early block 4 output prefix | relative L2: 0.0 | changed components: 0/65536
early block 4 output ima

## What the token-role analysis means

**All sampled early-stage prefix states match exactly; sampled image states differ.** At each candidate block 2?5, both input and output prefix comparisons have zero relative L2 difference and zero changed components out of **65,536**. Those comparisons cover **16 sampled prefix tokens ? 4096 features**, not every prefix token in the sequence.

The sampled image input to block 2 differs by **0.0626%** at early, **0.0309%** at middle and **0.0487%** at late. Early block-2 image input has **390,785 changed components out of 1,048,576** (256 tokens ? 4096 features). By block 5's output, early image relative error is approximately **0.2751%**. These are hidden-state errors, not velocity or image-quality measurements.

Prefix exact agreement argues against a wholesale weight-loading error affecting all tokens. The prefix uses zero-timestep conditioning and a different attention segment; image tokens use the diffusion timestep and attend over a much longer sequence. This result motivates testing attention execution, while leaving timestep-dependent computation and other numerical differences as alternatives. It does not identify the first differing operation or establish an attention-kernel root cause.

The analysis used saved tensors only: **zero teacher calls, zero student calls and zero optimizer updates**. The source checksums and all role-separated measurements are saved in `data/phase6/role_difference_analysis.json`. Its rerun reads the saved metrics without tensor loading or recomputation. Original captures and diagnostics remain unchanged.


## 6. Probe attention execution on the same teacher inputs

The saved role analysis found exactly matching sampled prefix states but differing image states. We test a controlled numerical hypothesis on the intact teacher: default SDPA selection, forced Flash Attention, memory-efficient attention, cuDNN attention and math attention. A sixth variant keeps default attention but disables BF16 reduced-precision matmul reduction. All variants use the same loaded model, saved conditioning, latents, timesteps and BF16 dtype; each rebuilds its own prefix cache.

For each supported variant we run early, middle and late inputs and compare full image-token velocities with their unchanged saved teacher targets. Unsupported backend errors are preserved as diagnostic results. We do not change the 0.001 relative L2 gate or insert a bridge. Every completed or partial variant's measured outputs and report are committed remotely and downloaded. This is a one-prompt causal probe; a matching setting would still need verification across all 30 cached prompts and a fresh model process. Reruns reuse the saved experiment without new forwards.


In [6]:
# Probe attention backends against the same saved teacher targets.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/attention_probe_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-attention-backend-probe-v1',
    'handoff_manifest_sha256': digest(handoff_path),
    'teacher': handoff['teacher'], 'record': handoff['target_index']['validation'][0],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'variants': ['default', 'flash', 'efficient', 'cudnn', 'math', 'default_full_reduction'],
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'},
    'control_relative_l2_tolerance': 0.001,
    'hypothesis': 'sampled prefix states match exactly but image attention differs; isolate SDPA backend and BF16 reduction setting',
    'policy': 'same intact teacher and exact saved inputs; fresh prefix cache per variant; no student, weights, dtype, inputs or tolerance changes; retain unsupported-backend failures and all measured outputs',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/attention_probe')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/attention_probe/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'probe_complete':
        raise RuntimeError('Attention probe incomplete or failed: ' + report.get('error', 'unknown'))
    if report['student_calls'] != 0 or report['optimizer_updates'] != 0 or len(report['variants']) != 6:
        raise RuntimeError('Unexpected probe counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-attention-probe')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['probe_attention'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Attention probe:', report['status'], '| GPU:', report['gpu'], '| teacher calls:', report['teacher_calls'])
for variant in report['variants']:
    print(variant['name'], variant['status'], '| controls passed:', variant.get('control_passed'), '| error:', variant.get('error'))
    for row in variant['controls']:
        print(' ', row['stage'], '| relative L2:', round(row['relative_l2_error'], 8), '| exact:', row['exact_equal'])
print('Saved results:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Attention probe: probe_complete | GPU: NVIDIA H100 80GB HBM3 | teacher calls: 15
default complete | controls passed: False | error: None
  early | relative L2: 0.00419176 | exact: False
  middle | relative L2: 0.00314055 | exact: False
  late | relative L2: 0.02385053 | exact: False
flash failed | controls passed: None | error: RuntimeError: No available kernel. Aborting execution.
efficient complete | controls passed: False | error: None
  early | relative L2: 0.00461827 | exact: False
  middle | relative L2: 0.00316706 | exact: False
  late | relative L2: 0.02516761 | exact: False
cudnn complete | controls passed: False | error: None
  early | relative L2: 0.00419176 | exact: False
  middle | relative L2: 0.00314055 | exact: False
  late | relative L2: 0.02385053 | exact: False
math complete | controls passed: False | error: None
  early | relative L2: 0.00629117 | exact: False
  middle | relative L2: 0.003

## What the attention probe means

**None of the whole-model settings reproduced all three saved velocities within the 0.1% tolerance.** The probe completed 15 teacher forwards across five supported variants; forced Flash Attention failed before its first complete forward because the prefix path supplies an attention mask unsupported by that kernel in this runtime. That error is a diagnostic result, not evidence that Flash cannot execute an unmasked image path.

| Variant | Early relative L2 | Middle relative L2 | Late relative L2 |
|---|---:|---:|---:|
| default | 0.4192% | 0.3141% | 2.3851% |
| efficient | 0.4618% | 0.3167% | 2.5168% |
| cudnn | 0.4192% | 0.3141% | 2.3851% |
| math | 0.6291% | 0.3177% | 2.5987% |
| default_full_reduction | 0.4192% | 0.3141% | 2.3851% |

Default selection and forced cuDNN produced identical reported errors. Disabling BF16 reduced-precision matmul reduction also left the reported errors unchanged. Forced efficient and math attention changed the numerical result but did not solve the mismatch. These observations narrow the tested settings; they do not identify the capture's per-segment kernel selection. The unchanged tolerance and saved targets remain the gate.

The pinned processor runs separate prefix and image attention calls. A whole-model Flash context cannot test the unmasked image call because it fails first on a masked prefix call. A targeted per-call experiment can test that remaining explanation without changing prefix computation. This is a diagnostic observation, not a claim that the capture used a particular kernel.

**Zero students and zero optimizer updates** ran. All supported variant velocities, the unsupported variant error, and the immutable request/report are saved under `data/phase6/attention_probe/` and backed up remotely. The original captures remain unchanged. Reruns read these saved outcomes without forwards or network calls.


## 7. Test Flash only on unmasked attention calls

The whole-model Flash probe could not reach image attention because masked prefix calls failed. We now wrap the pinned module's attention dispatch for this experiment only: masked calls use the unchanged original dispatch, while unmasked calls run under a Flash-only SDPA context. Original model weights, saved inputs, BF16 dtype and the 0.001 replay tolerance stay unchanged. We record which call shapes take each route.

We test the first validation prompt's three saved stages. If those pass, the same experiment verifies all 30 cached prompts, with fresh prefix caches and no bridges. It stops at the first failing stage and preserves every measured velocity locally and remotely. The original implementation and prior experiments are not edited. Passing this experiment would establish a reproducible per-call configuration for the cached targets, rather than merely a closer alternative answer. Reruns read saved results without new forwards.


In [7]:
# Test the original capture environment and verify saved teacher controls.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/segment_attention_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-unmasked-flash-verification-v1',
    'handoff_manifest_sha256': digest(handoff_path),
    'teacher': handoff['teacher'],
    'records': [handoff['target_index']['validation'][0]] + handoff['target_index']['train'] + handoff['target_index']['validation'][1:],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'control_relative_l2_tolerance': 0.001,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'},
    'hypothesis': 'preserve masked prefix calls and force Flash only for unmasked attention; test per-segment kernel selection against unchanged captured velocities',
    'policy': 'verify first failing prompt then all 30 cached prompts if controls pass; fresh teacher prefix cache per prompt; stop at first failed stage; persist each measured output',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/segment_attention')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/segment_attention/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Verification provenance differs.')
    if report['status'] not in ('passed', 'control_mismatch'):
        raise RuntimeError('Runtime verification failed; preserve attempt: ' + report.get('error', 'unknown'))
    if report['student_calls'] != 0 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected student or optimizer use.')
    if report['status'] == 'passed' and report['teacher_calls'] != 90:
        raise RuntimeError('Passing verification did not cover all 90 controls.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-unmasked-flash-verification')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['verify_segment_attention'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Unmasked Flash verification:', report['status'], '| GPU:', report['gpu'])
print('CUBLAS_WORKSPACE_CONFIG:', report['cublas_workspace_config'])
print('Teacher calls:', report['teacher_calls'], '| students:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Completed prompts:', report['completed_prompts'], '| exactly equal controls:', report['exact_equal_controls'])
print('Maximum relative L2 difference:', report['max_relative_l2_error'])
print('Saved report and outputs:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Unmasked Flash verification: control_mismatch | GPU: NVIDIA H100 80GB HBM3
CUBLAS_WORKSPACE_CONFIG: None
Teacher calls: 1 | students: 0 | updates: 0
Completed prompts: 1 | exactly equal controls: 0
Maximum relative L2 difference: 0.003878689371049404
Saved report and outputs: data\phase6\segment_attention


## What the per-call Flash test means

**Forcing Flash on unmasked attention did not restore the saved teacher prediction.** The first early-stage validation control differed by **0.3878689%** against the unchanged 0.1% tolerance. That is somewhat closer than the earlier default result but still fails the control. The worker stopped after one complete teacher forward, before middle/late stages or other prompts. It ran zero students and zero optimizer updates.

The route log records 32 masked prefix attention calls through original dispatch and 32 unmasked image attention calls through Flash. This demonstrates that the targeted experiment reached image attention despite the whole-model Flash failure. It does not establish which kernel the original capture used. Changing a backend to obtain a slightly smaller error is not a passing replay solution.

The prediction, route counts, immutable request and negative report are saved locally under `data/phase6/segment_attention/` and backed up remotely. Original captures and earlier diagnostic artifacts remain unchanged. The next investigation can use the role distinction to examine the timestep-dependent computation, because prefix text uses timestep zero while image tokens use the measured diffusion timestep. No root cause has been confirmed.


## 8. Probe timestep conditioning and original call reconstruction

Image-only differences are also compatible with a timestep-dependent discrepancy: prefix text always uses timestep zero. We reconstruct the pinned pipeline's full argument order and transfer every saved stage's fields inside inference mode, matching the original capture replay rather than sharing already-transferred fields.

On that baseline, we independently test a BF16 timestep-frequency buffer, CPU evaluation of the timestep trigonometric embedding, canonical CPU reconstruction of the RoPE tables, and a final replay after encoding one novel diagnostic prompt. The novel prompt's embeddings are saved; it is not a repeated dataset capture or a training example. Buffer and method changes are restored between variants. Original teacher weights, input tensors and targets stay unchanged; these are experiments, not accepted model changes.

Each variant compares the three stages of the first validation prompt against the original 0.001 relative L2 gate. We preserve velocities and timestep embeddings for every variant, including failed experiments, locally and remotely. There are no students or optimizer updates. Any matching variant must still pass all cached prompts in a separate fresh-process verification. Reruns reuse saved results without new model or encoder calls.


In [8]:
# Probe timestep computation, call reconstruction and pipeline warmup.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/conditioning_probe_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-conditioning-and-call-reconstruction-probe-v1',
    'handoff_manifest_sha256': digest(handoff_path), 'teacher': handoff['teacher'],
    'record': handoff['target_index']['validation'][0],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'variants': ['original_argument_order', 'bf16_frequency_buffer', 'cpu_timestep_embedding', 'canonical_cpu_rope', 'after_novel_text_warmup'],
    'warmup_prompt': 'A single crimson calibration cube on a neutral gray background, runtime diagnostic only.',
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'},
    'control_relative_l2_tolerance': 0.001,
    'policy': 'restore original full argument order and per-stage device copies; isolate temporal frequency dtype, CPU timestep calculation, CPU RoPE table reconstruction and a novel prompt encoder warmup; preserve every output; no students or altered targets',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/conditioning_probe')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/conditioning_probe/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'probe_complete':
        raise RuntimeError('Attention probe incomplete or failed: ' + report.get('error', 'unknown'))
    if report['student_calls'] != 0 or report['optimizer_updates'] != 0 or len(report['variants']) != 5:
        raise RuntimeError('Unexpected probe counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-conditioning-probe')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['probe_conditioning'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Conditioning probe:', report['status'], '| GPU:', report['gpu'], '| teacher calls:', report['teacher_calls'])
for variant in report['variants']:
    print(variant['name'], variant['status'], '| controls passed:', variant.get('control_passed'), '| error:', variant.get('error'))
    for row in variant['controls']:
        print(' ', row['stage'], '| relative L2:', round(row['relative_l2_error'], 8), '| exact:', row['exact_equal'])
print('Saved results:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Conditioning probe: probe_complete | GPU: NVIDIA H100 80GB HBM3 | teacher calls: 15
original_argument_order complete | controls passed: False | error: None
  early | relative L2: 0.00419176 | exact: False
  middle | relative L2: 0.00314055 | exact: False
  late | relative L2: 0.02385053 | exact: False
bf16_frequency_buffer complete | controls passed: False | error: None
  early | relative L2: 0.09801684 | exact: False
  middle | relative L2: 0.00455 | exact: False
  late | relative L2: 0.03277948 | exact: False
cpu_timestep_embedding complete | controls passed: False | error: None
  early | relative L2: 0.00419176 | exact: False
  middle | relative L2: 0.00314055 | exact: False
  late | relative L2: 0.02385053 | exact: False
canonical_cpu_rope complete | controls passed: False | error: None
  early | relative L2: 0.00419176 | exact: False
  middle | relative L2: 0.00314055 | exact: False
  late | relative L2:

## What the conditioning and reconstruction probe means

**None of the tested variants restored the legacy cached velocities within tolerance.** All five variants completed the three saved teacher stages, for **15 teacher forwards**, plus **one text-encoder call on a novel diagnostic prompt**. No dataset prompt was re-encoded, no trajectory was regenerated, and no student or optimizer ran.

| Variant | Early relative L2 | Middle relative L2 | Late relative L2 |
|---|---:|---:|---:|
| original_argument_order | 0.4192% | 0.3141% | 2.3851% |
| bf16_frequency_buffer | 9.8017% | 0.4550% | 3.2779% |
| cpu_timestep_embedding | 0.4192% | 0.3141% | 2.3851% |
| canonical_cpu_rope | 0.4192% | 0.3141% | 2.3851% |
| after_novel_text_warmup | 0.4192% | 0.3141% | 2.3851% |

Restoring argument order and per-stage copies, CPU timestep evaluation, canonical CPU RoPE tables, and the novel text warmup did not change the reported baseline errors. Casting the timestep-frequency buffer to BF16 made early error much larger, approximately **9.80%**; that experimental cast is rejected and the original FP32 buffer was restored. These results narrow the tested causes but do not locate the exact first differing operation in the original capture.

Original teacher controls were checked before serialization in the capture process. Fresh-process comparisons have now repeatedly failed despite matching saved values and implementation checks. The old cache's numerical execution contract does not contain enough evidence to explain this difference. That observation is not permission to silently overwrite the old targets or report them as reproduced.

**Decision supported:** test an explicitly deterministic reference in its own versioned namespace, then verify it in a separately loaded process before using it for Phase 6 comparisons. The frozen teacher architecture, weights and saved input samples remain the source of supervision. A new reference would be an explicit arithmetic-contract change, not a repair of the old cache or proof that the original discrepancy's root cause is known. The 0.1% gate stays unchanged.

All measured velocities, timestep embeddings and novel warmup embeddings are saved locally and remotely under `data/phase6/conditioning_probe/`. Prior artifacts remain unchanged. This cell's completed rerun reads saved outcomes without repeated calls or metric calculations.


## 9. Create a versioned deterministic teacher reference

The old captures remain preserved and their fresh-process mismatch remains unresolved. We now test a practical replay contract: use the same frozen teacher weights and exact saved input samples with deterministic algorithms, an explicit math SDPA backend, highest FP32 matmul precision, TF32 disabled, cuDNN benchmarking disabled and reduced-precision reductions disabled. `CUBLAS_WORKSPACE_CONFIG=:4096:8` is set before the process starts as part of this new deterministic contract, not asserted as the fix for the old mismatch.

We generate exactly 90 velocities from the 30 saved prompts and three stages, with a fresh prefix cache per prompt. This is a new arithmetic variant in a separate namespace. It does not replace original Phase 5 velocities, hidden pairs, bridge checkpoints, trajectories or metadata. We preserve its differences from the original velocities instead of reporting the old targets as reproduced.

Each prompt's outputs and provenance are committed remotely and downloaded with checksum verification. This cell alone cannot establish reproducibility: an independent fresh-model process must reproduce every new reference under the unchanged 0.001 tolerance before accepting the contract. There are no students, gradients, images or benchmarks. Reruns read saved generation results without calls or tensor calculations. Math SDPA may be slower than fused attention; this is an accuracy/replay contract, not an inference-speed claim.

PyTorch documents backend-specific SDPA numerical behavior and deterministic execution controls in its [reproducibility notes](https://github.com/pytorch/pytorch/blob/main/docs/source/notes/randomness.md). These notes motivate an explicit contract; they do not identify the root cause of this project's legacy-cache discrepancy.


In [9]:
# Create a separately versioned deterministic teacher reference.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/deterministic_reference_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-deterministic-teacher-reference-v1', 'mode': 'generate',
    'handoff_manifest_sha256': digest(handoff_path), 'teacher': handoff['teacher'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'},
    'arithmetic_contract': {
        'weights_dtype': 'bfloat16', 'attention_backend': 'SDPBackend.MATH',
        'deterministic_algorithms': True, 'float32_matmul_precision': 'highest',
        'tf32_matmul': False, 'tf32_cudnn': False, 'cudnn_benchmark': False, 'cudnn_deterministic': True,
        'bf16_reduced_precision_matmul_reduction': False, 'fp16_bf16_reduction_math_sdp': False,
        'input_order': 'original pipeline argument order; stage tensors copied inside inference mode',
        'prefix_cache': 'fresh teacher cache per prompt; populate at early; use at middle and late',
    },
    'policy': 'new versioned teacher velocities from unchanged saved input samples; preserve all legacy targets and compare against them; no bridge or gradient; verification must use a fresh process and original 0.001 gate',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/deterministic_reference_generation')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/deterministic_reference/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'reference_generated':
        raise RuntimeError('Deterministic generation incomplete: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 90 or report['student_calls'] != 0 or report['optimizer_updates'] != 0 or len(report['artifacts']) != 30:
        raise RuntimeError('Unexpected deterministic reference counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-deterministic-reference-generation')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['deterministic_reference'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Deterministic reference:', report['status'], '| GPU:', report['gpu'])
print('Teacher forwards:', report['teacher_calls'], '| student forwards:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Saved prompts:', len(report['artifacts']), '| maximum difference from legacy:', report['legacy_max_relative_l2'])
print('Generation process:', report['process'])
print('Saved outputs:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Deterministic reference: reference_generated | GPU: NVIDIA H100 80GB HBM3
Teacher forwards: 90 | student forwards: 0 | updates: 0
Saved prompts: 30 | maximum difference from legacy: 0.025987252593040466
Generation process: {'pid': 2, 'host': 'modal', 'nonce': '613d8c36de054bdcb9d0684cc4e6f788', 'modal_task_id': 'ta-01M3PAVXZV5XS0D0MHJ78F5TQR'}
Saved outputs: data\phase6\deterministic_reference_generation


## What the deterministic reference generation means

**A new numerical reference now exists for all 30 saved prompts and 90 stage inputs.** The frozen 32-block teacher used its unchanged checkpoint, BF16 weights and exact saved conditioning, latent and timestep samples. The arithmetic contract fixes math SDPA, deterministic algorithms, highest FP32 matmul precision and the recorded reduction/runtime settings. It creates a separately versioned cache; no original capture or bridge weight was replaced.

This stage executed **90 teacher forwards, zero student forwards and zero optimizer updates**. No text encoder, full denoising trajectory, image decoder, training or benchmark ran. Each prompt's early replay populates its own prefix cache for the middle and late calls.

### Explicit differences from the legacy teacher cache

These measurements compare the new math-runtime velocities with the original Phase 5 velocities. They are not verification errors for the new reference and must not be reported as evidence that the old cache was reproduced.

| Split | Stage | Prompts | Mean relative L2 difference from legacy | Maximum relative L2 difference from legacy |
|---|---|---:|---:|---:|
| train | early | 20 | 0.5784% | 0.8197% |
| train | middle | 20 | 0.3752% | 0.4936% |
| train | late | 20 | 0.9752% | 1.6989% |
| validation | early | 10 | 0.5691% | 0.7746% |
| validation | middle | 10 | 0.4017% | 0.5299% |
| validation | late | 10 | 1.1719% | 2.5987% |

The maximum difference from any legacy target is **2.5987%**. It exceeds the original 0.1% replay gate, so legacy numerical equivalence remains unestablished. This is an explicit arithmetic-variant change, while retaining the same teacher architecture, learned weights and sampled inputs. The metric describes model-output vectors, not generated-image quality. The Phase 5 bridges remain the original trained checkpoints; no claim about their performance under the new runtime has been made.

**Decision supported:** verify this fully specified reference from a fresh teacher process. **Still uncertain:** whether all 90 new outputs are reproducible across that second load, the exact cause of the legacy mismatch, and effects on complete-student predictions or images. `reference_generated` alone is not a passing reproducibility test.

The immutable request, report and 30 per-prompt velocity bundles are saved in `data/phase6/deterministic_reference_generation/` and backed up remotely. The report records Torch/CUDA/cuDNN/package versions and a distinct Modal task ID. All downloads passed checksum verification. A completed rerun uses saved artifacts without new model or network calls, calculations or replacements. Original Phase 4/5 targets and all diagnostic attempts remain preserved.


## 10. Verify the deterministic reference in a fresh model process

We start a separate Modal app and load the pinned teacher again under the identical deterministic arithmetic contract. This process replays the same 90 saved stage inputs and compares its outputs with the separate reference generated above. Original Phase 5 outputs remain the historical cache and their differences are still reported. The unchanged 0.001 relative L2 gate now validates the explicitly versioned reference; it is not loosened to accept a legacy mismatch.

The worker verifies reference/input checksums and matching package, CUDA and cuDNN versions. We also require distinct Modal task IDs, establishing separate worker tasks rather than repeated forwards in the same loaded model. Every measured verification velocity is preserved remotely and downloaded. Only after all 90 controls pass do we save `data/phase6/replay_contract.json`, binding inputs, teacher, arithmetic settings, packages, reference artifacts and verification evidence.

The contract requires future teacher/student comparisons to use the same arithmetic settings and their own prefix caches. It does not mix historical hidden pairs with new full-healing hidden supervision. No student, optimizer, trajectory, image or benchmark is run. Reruns read and verify saved files without forwards or network imports. We stop investigation work once this independent reproducibility check provides a working replay contract; the precise legacy discrepancy is recorded as unresolved.


In [10]:
# Independently verify the deterministic reference and preserve its replay contract.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/deterministic_reference_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-deterministic-teacher-reference-v1', 'mode': 'verify',
    'handoff_manifest_sha256': digest(handoff_path), 'teacher': handoff['teacher'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'},
    'arithmetic_contract': {
        'weights_dtype': 'bfloat16', 'attention_backend': 'SDPBackend.MATH',
        'deterministic_algorithms': True, 'float32_matmul_precision': 'highest',
        'tf32_matmul': False, 'tf32_cudnn': False, 'cudnn_benchmark': False, 'cudnn_deterministic': True,
        'bf16_reduced_precision_matmul_reduction': False, 'fp16_bf16_reduction_math_sdp': False,
        'input_order': 'original pipeline argument order; stage tensors copied inside inference mode',
        'prefix_cache': 'fresh teacher cache per prompt; populate at early; use at middle and late',
    },
    'reference_key': read_json('data/phase6/deterministic_reference_generation/request.json')['request_key'],
    'reference_report_sha256': digest('data/phase6/deterministic_reference_generation/summary.json'),
    'policy': 'new versioned teacher velocities from unchanged saved input samples; preserve all legacy targets and compare against them; no bridge or gradient; verification must use a fresh process and original 0.001 gate',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/deterministic_reference_verification')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/deterministic_reference/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'reference_verified':
        raise RuntimeError('Deterministic generation incomplete: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 90 or report['student_calls'] != 0 or report['optimizer_updates'] != 0 or len(report['artifacts']) != 30:
        raise RuntimeError('Unexpected deterministic reference counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-deterministic-reference-verification')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['deterministic_reference'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Deterministic reference:', report['status'], '| GPU:', report['gpu'])
print('Teacher forwards:', report['teacher_calls'], '| student forwards:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Saved prompts:', len(report['artifacts']), '| maximum difference from legacy:', report['legacy_max_relative_l2'])
print('Generation process:', report['process'])
print('Saved outputs:', run_dir)

generation_dir = Path('data/phase6/deterministic_reference_generation')
generation = read_json(generation_dir / 'summary.json')
first_task = generation['process']['modal_task_id']
second_task = report['process']['modal_task_id']
if not first_task or not second_task or first_task == second_task:
    raise RuntimeError('Independent Modal tasks were not established.')
if report['max_reference_relative_l2'] > 0.001:
    raise RuntimeError('Independent verification exceeds the unchanged gate.')
for artifact in generation['artifacts']:
    if digest(generation_dir / artifact['name']) != artifact['sha256']:
        raise RuntimeError('Local deterministic reference differs.')
contract = {
    'version': 'phase6-verified-replay-contract-v1',
    'status': 'verified_for_saved_pilot',
    'legacy_cache_reproduced': False,
    'handoff_manifest_sha256': digest(handoff_path),
    'generation_request_key': generation['request_key'],
    'verification_request_key': key,
    'generation_report_sha256': digest(generation_dir / 'summary.json'),
    'verification_report_sha256': digest(report_path),
    'teacher': request['teacher'],
    'arithmetic_contract': request['arithmetic_contract'],
    'runtime_environment': request['runtime_environment'],
    'packages': report['packages'], 'torch_cuda': report['torch_cuda'], 'cudnn_version': report['cudnn_version'],
    'teacher_velocity_directory': generation_dir.as_posix(),
    'reference_artifacts': generation['artifacts'],
    'input_records': request['records'],
    'control_relative_l2_tolerance': 0.001,
    'verified_controls': 90, 'max_reference_relative_l2': report['max_reference_relative_l2'],
    'exact_equal_controls': report['exact_equal_controls'],
    'independent_tasks': [first_task, second_task],
    'student_policy': 'teacher/student comparisons must use the same deterministic arithmetic contract and exact saved inputs; discard old prefix caches after bridge insertion',
    'hidden_supervision_policy': 'legacy Phase 5 hidden pairs remain historical bridge-fitting data; surviving-layer hidden targets for healing need matching-contract capture',
    'scope': 'cached 20 TRAIN / 10 VALIDATION pilot and three stages; no student, healing, image quality or speed benchmark established',
    'legacy_policy': 'preserve original inputs, velocities, hidden pairs, trained bridges and failed attempts; legacy mismatch remains unexplained',
}
contract_path = Path('data/phase6/replay_contract.json')
if contract_path.exists():
    if read_json(contract_path) != contract:
        raise RuntimeError('Verified replay contract changed; preserve existing result.')
else:
    with contract_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(contract, indent=2) + '\n')
print('Independent replay controls:', len(report['controls']), '| exactly equal:', report['exact_equal_controls'])
print('Maximum independent replay relative L2:', report['max_reference_relative_l2'])
print('Verified replay contract:', contract_path)


RuntimeError: Deterministic generation incomplete: RuntimeError: Generation report differs.

## What the first independent-verification attempt means

**Verification stopped on report provenance before any teacher forward.** The worker rejected the generation report's raw-file SHA-256. The local coordinator reserialized that JSON report using Windows text-mode line endings, while the remote report uses Unix line endings. Raw bytes therefore differ even when JSON content is identical. This bookkeeping gate is distinct from the legacy numerical mismatch.

The failed report records **zero teacher calls, zero student calls and zero optimizer updates**. It produced no verification velocities and established no cross-process reproducibility result. The generation reference and its 90 outputs remain unchanged. This attempt is saved under `data/phase6/deterministic_reference_verification/`.

The correction is to bind the report's canonical sorted JSON content for remote/local provenance comparison, while retaining checksums of every binary velocity artifact. The failed request and report are preserved; a separately versioned verifier will perform that corrected check. Rerunning this cell reads the saved failure without another GPU or model call.


## 11. Verify the reference with canonical report provenance

The prior verification failed before replay because raw JSON bytes differed across Windows and Unix line endings. This new verifier compares the SHA-256 of canonical sorted JSON content, retains binary checksums for every tensor artifact, and explicitly checks that generation and verification use identical arithmetic settings. The failed attempt is preserved unchanged.

We repeat the independent-process verification under that corrected provenance contract: a new Modal task, fresh teacher load, identical saved 90 inputs, deterministic math attention and the unchanged 0.001 relative L2 gate. All verification outputs are saved in a new namespace. The replay contract is written only after all controls pass and distinct Modal task IDs are confirmed. No student evaluation or training follows a passing verification; that is the stopping point requested in this session.


In [11]:
# Independently verify the deterministic reference and preserve its replay contract.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/deterministic_verification_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-deterministic-teacher-verification-v2', 'mode': 'verify',
    'handoff_manifest_sha256': digest(handoff_path), 'teacher': handoff['teacher'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'},
    'arithmetic_contract': {
        'weights_dtype': 'bfloat16', 'attention_backend': 'SDPBackend.MATH',
        'deterministic_algorithms': True, 'float32_matmul_precision': 'highest',
        'tf32_matmul': False, 'tf32_cudnn': False, 'cudnn_benchmark': False, 'cudnn_deterministic': True,
        'bf16_reduced_precision_matmul_reduction': False, 'fp16_bf16_reduction_math_sdp': False,
        'input_order': 'original pipeline argument order; stage tensors copied inside inference mode',
        'prefix_cache': 'fresh teacher cache per prompt; populate at early; use at middle and late',
    },
    'reference_key': read_json('data/phase6/deterministic_reference_generation/request.json')['request_key'],
    'reference_report_semantic_sha256': hashlib.sha256(json.dumps(read_json('data/phase6/deterministic_reference_generation/summary.json'), sort_keys=True).encode()).hexdigest(),
    'policy': 'new versioned teacher velocities from unchanged saved input samples; preserve all legacy targets and compare against them; no bridge or gradient; verification must use a fresh process and original 0.001 gate',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/deterministic_reference_verified')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/deterministic_reference/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'reference_verified':
        raise RuntimeError('Deterministic generation incomplete: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 90 or report['student_calls'] != 0 or report['optimizer_updates'] != 0 or len(report['artifacts']) != 30:
        raise RuntimeError('Unexpected deterministic reference counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-deterministic-reference-verification-v2')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['deterministic_reference'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Deterministic reference:', report['status'], '| GPU:', report['gpu'])
print('Teacher forwards:', report['teacher_calls'], '| student forwards:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Saved prompts:', len(report['artifacts']), '| maximum difference from legacy:', report['legacy_max_relative_l2'])
print('Generation process:', report['process'])
print('Saved outputs:', run_dir)

generation_dir = Path('data/phase6/deterministic_reference_generation')
generation = read_json(generation_dir / 'summary.json')
first_task = generation['process']['modal_task_id']
second_task = report['process']['modal_task_id']
if not first_task or not second_task or first_task == second_task:
    raise RuntimeError('Independent Modal tasks were not established.')
if report['max_reference_relative_l2'] > 0.001:
    raise RuntimeError('Independent verification exceeds the unchanged gate.')
for artifact in generation['artifacts']:
    if digest(generation_dir / artifact['name']) != artifact['sha256']:
        raise RuntimeError('Local deterministic reference differs.')
contract = {
    'version': 'phase6-verified-replay-contract-v1',
    'status': 'verified_for_saved_pilot',
    'legacy_cache_reproduced': False,
    'handoff_manifest_sha256': digest(handoff_path),
    'generation_request_key': generation['request_key'],
    'verification_request_key': key,
    'generation_report_sha256': digest(generation_dir / 'summary.json'),
    'generation_report_semantic_sha256': request['reference_report_semantic_sha256'],
    'verification_report_sha256': digest(report_path),
    'teacher': request['teacher'],
    'arithmetic_contract': request['arithmetic_contract'],
    'runtime_environment': request['runtime_environment'],
    'packages': report['packages'], 'torch_cuda': report['torch_cuda'], 'cudnn_version': report['cudnn_version'],
    'teacher_velocity_directory': generation_dir.as_posix(),
    'reference_artifacts': generation['artifacts'],
    'input_records': request['records'],
    'control_relative_l2_tolerance': 0.001,
    'verified_controls': 90, 'max_reference_relative_l2': report['max_reference_relative_l2'],
    'exact_equal_controls': report['exact_equal_controls'],
    'independent_tasks': [first_task, second_task],
    'student_policy': 'teacher/student comparisons must use the same deterministic arithmetic contract and exact saved inputs; discard old prefix caches after bridge insertion',
    'hidden_supervision_policy': 'legacy Phase 5 hidden pairs remain historical bridge-fitting data; surviving-layer hidden targets for healing need matching-contract capture',
    'scope': 'cached 20 TRAIN / 10 VALIDATION pilot and three stages; no student, healing, image quality or speed benchmark established',
    'legacy_policy': 'preserve original inputs, velocities, hidden pairs, trained bridges and failed attempts; legacy mismatch remains unexplained',
}
contract_path = Path('data/phase6/replay_contract.json')
if contract_path.exists():
    if read_json(contract_path) != contract:
        raise RuntimeError('Verified replay contract changed; preserve existing result.')
else:
    with contract_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(contract, indent=2) + '\n')
print('Independent replay controls:', len(report['controls']), '| exactly equal:', report['exact_equal_controls'])
print('Maximum independent replay relative L2:', report['max_reference_relative_l2'])
print('Verified replay contract:', contract_path)


RuntimeError: Deterministic generation incomplete: RuntimeError: Independent-process reference replay exceeds tolerance.

## What the canonical independent verification means

**Canonical report provenance passed, but the deterministic reference still failed across the fresh teacher load.** The separately loaded H100 teacher replayed the first TRAIN prompt at all three stages. Relative L2 differences from the newly generated deterministic reference were **0.4721% early**, **0.3761% middle** and **0.7459% late**, all above the unchanged **0.1%** tolerance. Package, CUDA, cuDNN and arithmetic-setting checks passed.

The attempt stopped after **three teacher forwards**, with zero students and zero optimizer updates. Both processes have distinct recorded Modal task IDs. This is evidence that enabling deterministic math attention and the tested runtime flags is insufficient for this replay; it is not a working fix. No active replay contract was written.

The new reference and all failed verification outputs remain separately preserved. No old cache, teacher input or bridge checkpoint was replaced. The difference may involve actual loaded or derived model tensors that are not covered by configuration/source hashes. A tensor-state audit can compare all loaded parameter bytes, registered buffers and the unregistered RoPE frequency tables before interpreting this as an execution-kernel issue. The exact root cause remains unresolved.


## 12. Audit actual loaded model tensors, first independent load

The explicit deterministic runtime also failed fresh-load verification, so it is not accepted. We now hash the raw bytes, shapes, dtypes and strides of every loaded teacher parameter and registered buffer. We also capture the RoPE frequency tables, which are ordinary tensor-list attributes rather than registered buffers and therefore were not covered by the earlier buffer check. Small derived constants are saved as tensors for direct comparison.

We record CPU information and a Modal task ID, then perform three teacher replays of the first TRAIN prompt under deterministic math attention. This is the first half of a state comparison; its fingerprint alone cannot establish equality across loads. No teacher weight is altered, no bridge is run, and original captures remain unchanged. Results are saved locally and remotely; reruns reuse fingerprints and measured outputs without rehashing model tensors or repeating forwards.


In [12]:
# Audit all loaded tensor bytes and preserve derived constants.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/model_state_audit_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-loaded-tensor-state-audit-v1', 'audit_id': 'A',
    'handoff_manifest_sha256': digest(handoff_path), 'teacher': handoff['teacher'],
    'record': handoff['target_index']['train'][0],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'},
    'previous_state': None,
    'policy': 'hash every loaded parameter, registered buffer and unregistered RoPE table; preserve actual constant tensors; three deterministic math teacher forwards; compare a second independent load before drawing conclusions',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/state_audit_A')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/state_audit/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'audit_complete':
        raise RuntimeError('Tensor audit incomplete: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 3 or report['student_calls'] != 0 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected audit counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-state-audit-A')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['audit_model_state'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Tensor-state audit:', report['status'], '| tensors:', len(report['state']))
print('State fingerprint:', report['state_sha256'], '| process:', report['process'])
print('Teacher calls:', report['teacher_calls'], '| students:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Differences from prior load:', list(report.get('state_differences', {})))
for row in report['controls']:
    print(row['stage'], '| legacy L2:', row['relative_l2_error'], '| velocity fingerprint:', row['velocity_sha256'])
print('Saved audit:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Tensor-state audit: audit_complete | tensors: 301
State fingerprint: 557150d2799845b3a69811384f2f5a654b42b01eb2a88b3138e6fee0e77137b2 | process: {'pid': 2, 'modal_task_id': 'ta-01M3PBKSMQYEACQGAYRNXZ2YZR'}
Teacher calls: 3 | students: 0 | updates: 0
Differences from prior load: []
early | legacy L2: 0.006110303103923798 | velocity fingerprint: 0f7597b3830936f940104667f73180432663b2e987621e532e8130f9c508f97c
middle | legacy L2: 0.0038824332877993584 | velocity fingerprint: 6a6101c296428e276e5b7cebb61165072ea2989a5314a34fa4e3c2fe244b4c7a
late | legacy L2: 0.008303661830723286 | velocity fingerprint: c4ba46abd2086d0483ec3c3adbf2d9f1ebf14af4df5d4ea9a05d18399c0c2734
Saved audit: data\phase6\state_audit_A


## What the first tensor-state audit means

**The first independent load now has fingerprints for 301 tensors:** 297 learned parameters, 1 registered buffer and 3 unregistered RoPE tables. These are counts of tensor objects, not scalar parameter counts. Hashes bind raw bytes, shapes, dtypes and strides. The saved constants preserve the actual frequency-table values, rather than merely their construction formula.

This audit performed three teacher forwards on the first TRAIN prompt under the recorded deterministic math settings. Their outputs still differ from legacy targets. A single fingerprint cannot establish that another loaded teacher has the same state, and the printed empty prior-difference list means no prior comparison was supplied, not that cross-load equality passed.

The fingerprints, CPU information, process ID, constants and velocities are saved locally and remotely under `data/phase6/state_audit_A/`. No teacher weight, bridge checkpoint or original target changed. A second independent load is needed to compare the 301 tensor fingerprints. Reruns read saved fingerprints without loading the model or rehashing its parameters.


## 13. Compare all loaded tensors with a second independent teacher load

We load the pinned teacher in a separate Modal task, hash the same 301 tensor objects and compare each fingerprint with audit A. This includes every learned parameter, the timestep-frequency buffer and all three unregistered RoPE tables. We preserve CPU information and actual constant tensors so any difference can be inspected directly.

We replay the same first TRAIN prompt's three stages under the same deterministic math settings and save its measured velocities. Tensor-state equality and velocity equality are separate observations: matching weights/configuration alone do not prove matching computation. This audit introduces no new teacher weights, student, optimizer or target replacement. Completed reruns read the saved state comparison and outputs without model calls or parameter hashing.


In [13]:
# Audit all loaded tensor bytes and preserve derived constants.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/model_state_audit_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-loaded-tensor-state-audit-v1', 'audit_id': 'B',
    'handoff_manifest_sha256': digest(handoff_path), 'teacher': handoff['teacher'],
    'record': handoff['target_index']['train'][0],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'},
    'previous_state': read_json('data/phase6/state_audit_A/summary.json')['state'],
    'policy': 'hash every loaded parameter, registered buffer and unregistered RoPE table; preserve actual constant tensors; three deterministic math teacher forwards; compare a second independent load before drawing conclusions',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/state_audit_B')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/state_audit/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'audit_complete':
        raise RuntimeError('Tensor audit incomplete: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 3 or report['student_calls'] != 0 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected audit counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-state-audit-B')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['audit_model_state'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Tensor-state audit:', report['status'], '| tensors:', len(report['state']))
print('State fingerprint:', report['state_sha256'], '| process:', report['process'])
print('Teacher calls:', report['teacher_calls'], '| students:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Differences from prior load:', list(report.get('state_differences', {})))
for row in report['controls']:
    print(row['stage'], '| legacy L2:', row['relative_l2_error'], '| velocity fingerprint:', row['velocity_sha256'])
print('Saved audit:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Tensor-state audit: audit_complete | tensors: 301
State fingerprint: 557150d2799845b3a69811384f2f5a654b42b01eb2a88b3138e6fee0e77137b2 | process: {'pid': 2, 'modal_task_id': 'ta-01M3PBS4PD7DQY3JRAWJE9XRFR'}
Teacher calls: 3 | students: 0 | updates: 0
Differences from prior load: []
early | legacy L2: 0.006110303103923798 | velocity fingerprint: 0f7597b3830936f940104667f73180432663b2e987621e532e8130f9c508f97c
middle | legacy L2: 0.0038824332877993584 | velocity fingerprint: 6a6101c296428e276e5b7cebb61165072ea2989a5314a34fa4e3c2fe244b4c7a
late | legacy L2: 0.008303661830723286 | velocity fingerprint: c4ba46abd2086d0483ec3c3adbf2d9f1ebf14af4df5d4ea9a05d18399c0c2734
Saved audit: data\phase6\state_audit_B


## What the second model-state audit means

**All 301 tensor fingerprints matched exactly across the two independent teacher loads.** This covers 297 learned parameter tensors, one registered timestep-frequency buffer and all three unregistered RoPE tables. There are zero differing entries, including raw bytes, shapes, dtypes and strides. The CPU records identify different CPU model numbers, yet these model-state bytes still match. This does not support derived-table or parameter corruption as the explanation for these two loads.

The early, middle and late velocity fingerprints also match exactly between audit A and audit B. Their legacy differences are approximately **0.6110%, 0.3882% and 0.8304%**, respectively. Those values match the deterministic reference generation's first prompt. This establishes a repeatable three-stage execution path across the audited loads, while the prior verifier produced different values. A three-stage match is not a 90-control verification.

No weights, buffers, inputs or targets were changed by the audit. Full parameter hashing entails reading GPU tensors back to CPU, and the audit replays before loading any external reference-output tensor. These execution/context differences are possible leads, not yet established causes. The next verification will use this observed successful path with full state validation, explicit synchronization, and reference comparisons performed after the prompt's predictions are computed.

The second audit ran three teacher forwards, zero students and zero optimizer updates. Its fingerprints, actual constants and velocity tensors are saved under `data/phase6/state_audit_B/` and backed up remotely. Original targets and all failed experiments remain preserved. No active replay contract has yet been accepted.


### Audit the complete teacher state and independently verify all saved reference outputs

The two fresh-load audits matched all 301 model-state tensors and all three first-prompt velocities. This step uses that successful execution sequence to replay all 30 saved prompts (20 TRAIN and 10 VALIDATION), three stages each, in another fresh process. It checks the complete model state before inference and loads comparison targets after each prompt replay. We keep the original 0.001 relative-L2 gate, preserve legacy results, and save a replay contract only if all 90 checks pass. Existing saved results are reused without another GPU run. This is a test of a reproducible execution path; it does not establish the cause of earlier differences.


In [14]:
# Independently verify the deterministic reference and preserve its replay contract.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
worker_path = Path('data/phase6/audited_reference_verification_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-audited-reference-verification-v3', 'mode': 'verify',
    'expected_state': read_json('data/phase6/state_audit_A/summary.json')['state'],
    'handoff_manifest_sha256': digest(handoff_path), 'teacher': handoff['teacher'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'worker_source': worker_source,
    'runtime_environment': {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'},
    'arithmetic_contract': {
        'weights_dtype': 'bfloat16', 'attention_backend': 'SDPBackend.MATH',
        'deterministic_algorithms': True, 'float32_matmul_precision': 'highest',
        'tf32_matmul': False, 'tf32_cudnn': False, 'cudnn_benchmark': False, 'cudnn_deterministic': True,
        'bf16_reduced_precision_matmul_reduction': False, 'fp16_bf16_reduction_math_sdp': False,
        'input_order': 'original pipeline argument order; stage tensors copied inside inference mode',
        'prefix_cache': 'fresh teacher cache per prompt; populate at early; use at middle and late',
    },
    'reference_key': read_json('data/phase6/deterministic_reference_generation/request.json')['request_key'],
    'reference_report_semantic_sha256': hashlib.sha256(json.dumps(read_json('data/phase6/deterministic_reference_generation/summary.json'), sort_keys=True).encode()).hexdigest(),
    'policy': 'new versioned teacher velocities from unchanged saved input samples; preserve all legacy targets and compare against them; no bridge or gradient; verification must use a fresh process and original 0.001 gate',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/audited_reference_verification')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/audited_reference_verification/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'reference_verified':
        raise RuntimeError('Deterministic generation incomplete: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 90 or report['student_calls'] != 0 or report['optimizer_updates'] != 0 or len(report['artifacts']) != 30:
        raise RuntimeError('Unexpected deterministic reference counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-audited-reference-verification-v3')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=1800,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['deterministic_reference'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Deterministic reference:', report['status'], '| GPU:', report['gpu'])
print('Teacher forwards:', report['teacher_calls'], '| student forwards:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Saved prompts:', len(report['artifacts']), '| maximum difference from legacy:', report['legacy_max_relative_l2'])
print('Generation process:', report['process'])
print('Saved outputs:', run_dir)

generation_dir = Path('data/phase6/deterministic_reference_generation')
generation = read_json(generation_dir / 'summary.json')
first_task = generation['process']['modal_task_id']
second_task = report['process']['modal_task_id']
if not first_task or not second_task or first_task == second_task:
    raise RuntimeError('Independent Modal tasks were not established.')
if report['max_reference_relative_l2'] > 0.001:
    raise RuntimeError('Independent verification exceeds the unchanged gate.')
for artifact in generation['artifacts']:
    if digest(generation_dir / artifact['name']) != artifact['sha256']:
        raise RuntimeError('Local deterministic reference differs.')
contract = {
    'version': 'phase6-audited-replay-contract-v2',
    'expected_model_state': request['expected_state'],
    'execution_policy': 'audit the complete model state before inference; load reference output bundles only after prompt replay',
    'status': 'verified_for_saved_pilot',
    'legacy_cache_reproduced': False,
    'handoff_manifest_sha256': digest(handoff_path),
    'generation_request_key': generation['request_key'],
    'verification_request_key': key,
    'generation_report_sha256': digest(generation_dir / 'summary.json'),
    'generation_report_semantic_sha256': request['reference_report_semantic_sha256'],
    'verification_report_sha256': digest(report_path),
    'teacher': request['teacher'],
    'arithmetic_contract': request['arithmetic_contract'],
    'runtime_environment': request['runtime_environment'],
    'packages': report['packages'], 'torch_cuda': report['torch_cuda'], 'cudnn_version': report['cudnn_version'],
    'teacher_velocity_directory': generation_dir.as_posix(),
    'reference_artifacts': generation['artifacts'],
    'input_records': request['records'],
    'control_relative_l2_tolerance': 0.001,
    'verified_controls': 90, 'max_reference_relative_l2': report['max_reference_relative_l2'],
    'exact_equal_controls': report['exact_equal_controls'],
    'independent_tasks': [first_task, second_task],
    'student_policy': 'teacher/student comparisons must use the same deterministic arithmetic contract and exact saved inputs; discard old prefix caches after bridge insertion',
    'hidden_supervision_policy': 'legacy Phase 5 hidden pairs remain historical bridge-fitting data; surviving-layer hidden targets for healing need matching-contract capture',
    'scope': 'cached 20 TRAIN / 10 VALIDATION pilot and three stages; no student, healing, image quality or speed benchmark established',
    'legacy_policy': 'preserve original inputs, velocities, hidden pairs, trained bridges and failed attempts; legacy mismatch remains unexplained',
}
contract_path = Path('data/phase6/replay_contract.json')
if contract_path.exists():
    if read_json(contract_path) != contract:
        raise RuntimeError('Verified replay contract changed; preserve existing result.')
else:
    with contract_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(contract, indent=2) + '\n')
print('Independent replay controls:', len(report['controls']), '| exactly equal:', report['exact_equal_controls'])
print('Maximum independent replay relative L2:', report['max_reference_relative_l2'])
print('Verified replay contract:', contract_path)


Reusing verified replay results; no model imports, network calls or forwards.
Deterministic reference: reference_verified | GPU: NVIDIA H100 80GB HBM3
Teacher forwards: 90 | student forwards: 0 | updates: 0
Saved prompts: 30 | maximum difference from legacy: 0.025987252593040466
Generation process: {'pid': 2, 'host': 'modal', 'nonce': 'd7287544f546439d9ab4e836f891b35f', 'modal_task_id': 'ta-01M3PC69K2Z0QTB25XNM9CDA1R'}
Saved outputs: data\phase6\audited_reference_verification
Independent replay controls: 90 | exactly equal: 90
Maximum independent replay relative L2: 0.0
Verified replay contract: data\phase6\replay_contract.json


### Result: independent replay matches all 90 saved reference outputs exactly

A fresh H100 process reproduced all three stages for every saved prompt: **90 out of 90 outputs were exactly equal**, with maximum relative L2 error **0.0** against the separately generated reference. Relative L2 measures the size of the output difference divided by the size of the reference output; our unchanged acceptance threshold is 0.001 (0.1%). The run also matched the audited state of all **301 model tensors** before inference. This supports using the recorded audited execution sequence for comparisons on these 20 TRAIN and 10 VALIDATION pilot prompts. It is evidence from these saved inputs, not a guarantee for arbitrary prompts or runtime changes.

We saved all 30 output bundles, their checksums, the verification report, and the explicit replay contract in `data/phase6/replay_contract.json`. A local rerun loaded the saved results and verified checksums without importing Modal, Torch, or Diffusers, so it made no model or network calls and recalculated no metrics.

The new reference still differs from the original Phase 5 velocities by up to **2.5987% relative L2**. The exact cause of that older mismatch remains unresolved; none of the original targets, hidden pairs, or fitted bridges was replaced. This result resolves the operational replay blocker by establishing a separate, verified reference and execution contract. It does not prove that simply loading the same teacher twice inherently changes its outputs: the successful fresh processes produced identical results.

This run performed **90 teacher forwards, 0 student forwards, and 0 optimizer updates**. Phase 6 student evaluation and healing have not been performed. We stop here after establishing the verified replay path, as requested.


### Conclusion: the setup that reproduced the teacher outputs

We found an execution sequence that reproduced the **new, separately saved teacher reference** exactly across fresh processes. All **90 comparisons** passed (30 prompts, three stages each), with maximum relative L2 error **0.0**, below the unchanged 0.001 threshold. The model-state audit matched all **301 tensors**. This verifies replay on the saved pilot; it does not establish the root cause of the older Phase 5 mismatch.

The successful setup used all of the following together:

- **A fixed model and runtime:** the pinned Qwen-Image-2.1 revision and transformer source, BF16 weights, an H100 GPU, and the recorded software versions. The complete pins and provenance are saved in `data/phase6/replay_contract.json`; a package version label alone is insufficient for the pinned Diffusers development build.
- **Explicit numerical settings:** PyTorch `SDPBackend.MATH` attention, deterministic algorithms enabled, highest FP32 matrix-multiplication precision, TF32 disabled, cuDNN benchmarking disabled and deterministic behavior enabled, and reduced-precision BF16 matrix-multiplication and math-attention reductions disabled. `CUBLAS_WORKSPACE_CONFIG=:4096:8` was set before the runtime started.
- **Identical saved inputs and call sequence:** preserve the original pipeline argument order, copy each stage's tensors to the GPU inside inference mode, and use a fresh prefix cache for each prompt. Populate that cache at the early stage and reuse it at the middle and late stages.
- **The audited verification sequence:** check the complete model state before inference, run the prompt's three predictions, and only then load the saved reference output bundle for comparison.

These details describe the successful procedure. **We have not isolated which details are necessary or sufficient.** In particular, deterministic numerical settings alone did not make every earlier verification attempt match. The model-state check and delayed loading of comparison targets are part of the verified procedure, but neither has been proved to cause the improvement.

The new reference differs from the original Phase 5 velocities by up to **2.5987% relative L2**. The original targets, hidden-state pairs, trained bridges, and unsuccessful diagnostic results remain preserved. Identical audited model states argue against a changed checkpoint in the successful fresh-load tests; the exact explanation for the historical discrepancy remains unknown.

Phase 6 has established a reproducible teacher baseline for the saved pilot. It has not yet evaluated a bridged student or performed architecture healing. The Phase 5 bridges remain existing pretrained candidates: their historical fitting results do not establish their quality under this new replay contract. Hidden-state supervision used for healing must come from a teacher capture made under the matching contract rather than silently reusing the historical hidden targets.


### Evaluate the existing bridges against the verified teacher reference

We test the four saved Phase 5 bridges independently, each replacing just its original block (2, 3, 4, or 5). An identity replacement provides the matched baseline for each candidate. Before inserting any bridge, a fresh audited teacher must reproduce all 90 saved reference outputs within the unchanged 0.001 relative-L2 threshold, following the verified numerical settings and call sequence.

The student comparisons use the 10 existing VALIDATION prompts at early, middle, and late stages: four candidates times two variants times 10 prompts times three stages, or 240 student forwards. Each variant builds its own prefix cache from its modified states. We save every output and per-prompt metric, including regressions, and compare only with the new teacher reference. Lower relative L2 means closer teacher velocity; the identity-to-trained reduction measures whether pretraining helps the complete student. These prompts were already used for Phase 5 checkpoint selection, so this is a validation comparison, not an independent test-set claim. No gradients or healing updates are performed. Saved results and checksums prevent repeat model calls on rerun.


In [15]:
# Compare independent BF16 students on cached validation inputs, with durable reuse.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
contract = read_json('data/phase6/replay_contract.json')
if contract['status'] != 'verified_for_saved_pilot' or contract['verified_controls'] != 90:
    raise RuntimeError('A verified replay contract is required.')
worker_path = Path('data/phase6/verified_student_replay_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-verified-student-replay-v2', 'mode': 'verify',
    'replay_contract_sha256': digest('data/phase6/replay_contract.json'),
    'expected_state': contract['expected_model_state'],
    'arithmetic_contract': contract['arithmetic_contract'],
    'runtime_environment': contract['runtime_environment'],
    'reference_key': contract['generation_request_key'],
    'reference_report_semantic_sha256': contract['generation_report_semantic_sha256'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'handoff_manifest_sha256': digest(handoff_path),
    'teacher': handoff['teacher'], 'candidates': handoff['candidates'],
    'validation': handoff['target_index']['validation'],
    'fit_key': handoff['phase5_fitting_request_key'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'core_source': Path('data/phase4/bridge_definition.py').read_text(encoding='utf-8'),
    'wrapper_source': Path('data/phase4/bridge_block_definition.py').read_text(encoding='utf-8'),
    'worker_source': worker_source,
    'variants': ['identity', 'trained'], 'stages': [0, 20, 39],
    'teacher_control_relative_l2_tolerance': 0.001,
    'precision': 'BF16 pipeline weights and residual addition; FP32 metrics',
    'cache_policy': 'fresh prefix cache for every teacher/candidate/variant/prompt; rebuild at extraction',
    'metrics': 'full target-image velocity relative L2, MSE, RMSE, maximum absolute difference and cosine; unweighted prompt means by stage',
    'selection': 'no winner selected; retain all per-prompt outputs and regressions',
    'scope': 'held-out fixed-input inference compatibility; no optimizer, gradient test, images or student trajectories',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/verified_student_replay')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/verified_student_replay/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Replay report provenance differs.')
    if report['status'] != 'passed':
        raise RuntimeError('Replay failed; inspect saved attempt, no automatic retry: ' + report.get('error', 'unknown'))
    if report['max_reference_relative_l2'] > 0.001 or len(report['artifacts']) != 40:
        raise RuntimeError('Teacher controls or artifact counts differ.')
    if report['teacher_calls'] != 90 or report['student_calls'] != 240 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected replay counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-verified-student-replay')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=3600,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['replay_students'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Student replay:', report['status'], '| GPU:', report['gpu'])
print('Teacher control calls:', report['teacher_calls'], '| student calls:', report['student_calls'], '| updates:', report['optimizer_updates'])
for row in report['aggregates']:
    print(row['candidate_id'], '|', row['stage'], '| identity L2:', round(row['identity_mean_relative_l2'], 6),
          '| trained L2:', round(row['trained_mean_relative_l2'], 6), '| improved prompts:', str(row['improved_comparisons']) + '/' + str(row['comparisons']))
print('Saved report and velocity tensors:', run_dir)

print('Teacher exact comparisons:', report['exact_equal_controls'], '| max reference relative L2:', report['max_reference_relative_l2'])


Reusing verified replay results; no model imports, network calls or forwards.
Student replay: passed | GPU: NVIDIA H100 80GB HBM3
Teacher control calls: 90 | student calls: 240 | updates: 0
single_bridge_block_02 | early | identity L2: 0.146345 | trained L2: 0.141747 | improved prompts: 7/10
single_bridge_block_02 | middle | identity L2: 0.044404 | trained L2: 0.040154 | improved prompts: 10/10
single_bridge_block_02 | late | identity L2: 0.195615 | trained L2: 0.178804 | improved prompts: 10/10
single_bridge_block_02 | all | identity L2: 0.128788 | trained L2: 0.120235 | improved prompts: 27/30
single_bridge_block_03 | early | identity L2: 0.160166 | trained L2: 0.162396 | improved prompts: 5/10
single_bridge_block_03 | middle | identity L2: 0.034774 | trained L2: 0.034787 | improved prompts: 5/10
single_bridge_block_03 | late | identity L2: 0.12416 | trained L2: 0.109971 | improved prompts: 10/10
single_bridge_block_03 | all | identity L2: 0.106366 | trained L2: 0.102384 | improved p

### Result: the existing bridges improve complete-student velocity agreement

All **90 intact-teacher controls matched exactly**, so these comparisons use the verified teacher reference. The run completed **240 student forwards and no optimizer updates**. Each row below averages relative L2 equally over 10 validation prompts and three stages. Relative L2 measures the norm of the velocity difference divided by the teacher velocity norm; lower is closer. The reduction compares the trained bridge with identity replacement of that same block, not with the intact teacher.

| Replaced block | Identity mean error | Trained mean error | Relative reduction vs identity | Improved prompt-stage comparisons |
| --- | --- | --- | --- | --- |
| 2 | 12.879% | 12.023% | 6.64% | 27 / 30 |
| 3 | 10.637% | 10.238% | 3.74% | 20 / 30 |
| 4 | 7.160% | 5.873% | 17.97% | 26 / 30 |
| 5 | 6.406% | 5.518% | 13.86% | 24 / 30 |

For block 2, the absolute decrease is about **0.855 percentage points**. The relative reduction divides that decrease by the identity error: **(12.8788036 - 12.0234947) / 12.8788036 ? 100 = 6.6412%**, displayed as **6.64%**. The table rounds the error columns separately; using only their displayed values gives about 6.65%. The 27 / 30 count means 27 of the 30 individual prompt-stage errors decreased.

Block **5** gives the closest average teacher velocity and the lowest worst-case error among these four trained candidates (15.095% relative L2). Block **4** gives the largest reduction from its identity baseline. These observations make block 5 a reasonable candidate for a healing pilot; they do not establish image quality or speed, and the small difference between blocks 4 and 5 has no uncertainty estimate here.

Pretraining is not uniformly beneficial. Block 3's early-stage mean increases from 16.017% to 16.240%, and its middle-stage mean increases slightly from 3.477% to 3.479%. Even block 5 regresses against identity in 6 of 30 prompt-stage comparisons. All individual outputs and metrics, including regressions, are preserved in `data/phase6/verified_student_replay` locally and remotely.

These are fixed teacher-input predictions: they do not measure accumulated error along a student-generated trajectory. The validation prompts were also used for Phase 5 checkpoint selection, so they are not an independent test set. The results support retaining the existing bridges as candidates rather than automatically repeating pretraining; a comparison on fresh matching-contract hidden pairs is still needed to assess local fitting under the revised teacher settings. A checksum-verified local rerun reused the saved results without importing Torch, Diffusers, or Modal.


### Capture matching-contract teacher hidden targets and reassess local bridge fitting

We replay all 20 TRAIN and 10 VALIDATION prompt inputs using the audited teacher settings, argument order, and fresh prefix caches. Hooks record the input and output of original blocks 2?5 at the exact same sampled token positions and roles used in Phase 5. This creates a separate teacher-target version while preserving every historical artifact. The hooks are removed after each prompt, and all 90 resulting teacher velocities must match the independently verified reference within the unchanged 0.001 relative-L2 gate before the capture is accepted.

After all teacher forwards finish, we evaluate the existing frozen bridges on the new hidden pairs with the original fitting arithmetic: FP32 parameters and residual addition, BF16 autocast projections, and the original equally weighted four-group objective (prompt tokens plus early, middle, and late image tokens). The objective adds squared distance between normalized vectors to 0.1 times per-token squared relative error. This isolates whether the bridges' local fit survives the changed teacher contract. We also save per-role differences between historical and new hidden states. No optimizer or full student forward is run in this step; completed results and checksums are reused on rerun.


In [16]:
# Compare independent BF16 students on cached validation inputs, with durable reuse.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
contract = read_json('data/phase6/replay_contract.json')
if contract['status'] != 'verified_for_saved_pilot' or contract['verified_controls'] != 90:
    raise RuntimeError('A verified replay contract is required.')
worker_path = Path('data/phase6/aligned_hidden_capture_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-aligned-hidden-capture-v1', 'mode': 'verify',
    'candidate_layers': [2, 3, 4, 5],
    'magnitude_weight': 0.1,
    'capture_policy': 'same saved inputs and exact historical token selections; refreshed teacher pairs and velocity controls; preserve legacy cache',
    'replay_contract_sha256': digest('data/phase6/replay_contract.json'),
    'expected_state': contract['expected_model_state'],
    'arithmetic_contract': contract['arithmetic_contract'],
    'runtime_environment': contract['runtime_environment'],
    'reference_key': contract['generation_request_key'],
    'reference_report_semantic_sha256': contract['generation_report_semantic_sha256'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'handoff_manifest_sha256': digest(handoff_path),
    'teacher': handoff['teacher'], 'candidates': handoff['candidates'],
    'validation': handoff['target_index']['validation'],
    'fit_key': handoff['phase5_fitting_request_key'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'core_source': Path('data/phase4/bridge_definition.py').read_text(encoding='utf-8'),
    'wrapper_source': Path('data/phase4/bridge_block_definition.py').read_text(encoding='utf-8'),
    'worker_source': worker_source,
    'variants': ['identity', 'trained'], 'stages': [0, 20, 39],
    'teacher_control_relative_l2_tolerance': 0.001,
    'precision': 'BF16 teacher; bridge diagnostics use original Phase 5 FP32 parameters/residual and BF16 autocast projections; FP32 losses',
    'cache_policy': 'fresh prefix cache for every teacher/candidate/variant/prompt; rebuild at extraction',
    'metrics': 'teacher velocity controls; hidden input/target differences by role; original Phase 5 balanced directional plus 0.1 relative squared objective on fresh pairs',
    'selection': 'no optimization; assess existing bridges on matching-contract pairs',
    'scope': 'fresh hidden pairs on all 30 pilot prompts; frozen bridge local-fit diagnostics only',
}
key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/aligned_hidden_capture')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/aligned_hidden_capture/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key:
        raise RuntimeError('Replay report provenance differs.')
    if report['status'] != 'passed':
        raise RuntimeError('Replay failed; inspect saved attempt, no automatic retry: ' + report.get('error', 'unknown'))
    if report['max_reference_relative_l2'] > 0.001 or len(report['artifacts']) != 30:
        raise RuntimeError('Teacher controls or artifact counts differ.')
    if report['teacher_calls'] != 90 or report['student_calls'] != 0 or report['optimizer_updates'] != 0:
        raise RuntimeError('Unexpected replay counts.')

complete = False
if report is not None:
    validate(report)
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        app = modal.App('nihonga-phase6-aligned-hidden-capture')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=3600,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['capture_hidden'])
        with modal.enable_output(), app.run():
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    validate(report)
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
print('Aligned hidden capture:', report['status'], '| GPU:', report['gpu'])
print('Teacher forwards:', report['teacher_calls'], '| full student forwards:', report['student_calls'], '| updates:', report['optimizer_updates'])
print('Teacher exact comparisons:', report['exact_equal_controls'], '| maximum relative L2:', report['max_reference_relative_l2'])
for row in report['bridge_hidden_aggregates']:
    print('Block', row['layer'], '|', row['split'], '| identity objective:', row['identity']['balanced_objective'],
          '| trained objective:', row['trained']['balanced_objective'], '| reduction:', row['relative_objective_reduction'])
print('Fresh pairs and report:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Aligned hidden capture: passed | GPU: NVIDIA H100 80GB HBM3
Teacher forwards: 90 | full student forwards: 0 | updates: 0
Teacher exact comparisons: 90 | maximum relative L2: 0.0
Block 2 | train | identity objective: 0.007435873156646267 | trained objective: 0.002093818777939305 | reduction: 0.7184165552813624
Block 2 | validation | identity objective: 0.00728594395914115 | trained objective: 0.0039686774194706235 | reduction: 0.4552967409951858
Block 3 | train | identity objective: 0.00821964634815231 | trained objective: 0.002835612601484172 | reduction: 0.6550201211367703
Block 3 | validation | identity objective: 0.008408732386305929 | trained objective: 0.005213093210477382 | reduction: 0.3800381590253503
Block 4 | train | identity objective: 0.009533440554514527 | trained objective: 0.0027627062023384496 | reduction: 0.7102089023851751
Block 4 | validation | identity objective: 0.010014398931525647 | tra

### Result: matching-contract hidden targets preserve the bridges' local fitting gains

The capture completed **90 teacher forwards**, and all **90 teacher velocity controls were exactly equal** to the verified reference (maximum relative L2 **0.0**). It recorded **360 block input/output pairs**: four blocks, three stages, and 30 prompts, using the same token positions and roles as the historical capture. All 30 refreshed bundles are saved locally and remotely, with their checksums and source-input provenance. The original Phase 5 files remain preserved.

The existing frozen bridges still improve the original, equally weighted four-group validation objective on these fresh targets:

| Block | Fresh identity objective | Fresh trained objective | Objective reduction |
| --- | --- | --- | --- |
| 2 | 0.0072859 | 0.0039687 | 45.53% |
| 3 | 0.0084087 | 0.0052131 | 38.00% |
| 4 | 0.0100144 | 0.0050370 | 49.70% |
| 5 | 0.0173872 | 0.0067119 | 61.40% |

The objective combines directional agreement and per-token relative squared error. Its numbers are dimensionless fitting scores, not percentages of velocity or image error; smaller is better. The reduction column compares each trained bridge with identity on that block's fresh targets. The **38?61% reductions** closely match the historical Phase 5 fitting gains, even though full-student velocity improvements are smaller. A local module can fit its hidden target well without recovering the complete model's behavior to the same extent.

For block 5, the trained validation objective is **0.006711873**, versus **0.006713707** in the historical Phase 5 evaluation, a change of about **-0.027%**. This provides no evidence that the replay change alone requires another Phase 5 fitting run. It does not prove that further bridge training could never help or explain the historical replay discrepancy.

The step used **0 optimizer updates and 0 full student forwards**; frozen bridge evaluations are local diagnostics, not architecture healing. TRAIN and VALIDATION results and per-role hidden differences are saved in the report. The checks establish velocity consistency for this capture; the hidden tensors themselves were not independently recaptured in a second process. A local cached rerun verified saved files without importing Modal, Torch, or Diffusers, making no network or model calls and recalculating no metrics.


### Preserve the bridge assessment and index the refreshed targets

We use the saved full-student velocity comparison and fresh hidden fitting results to choose an existing candidate for a healing pilot. The rule is the lowest equal-prompt, equal-stage mean validation velocity relative L2 among candidates whose trained bridge improves both velocity agreement and local hidden fitting versus identity. This is an operational choice on the small existing validation split, not a claim of statistical superiority.

We serialize the selected checkpoint and architecture recipe, all four candidates' evidence, report checksums, and a 20 TRAIN / 10 VALIDATION index pointing to the matching-contract target bundles. The first execution derives and saves this assessment; reruns read the saved decision and check provenance without repeating selection, metrics, teacher calls, or training. This records whether a Phase 5 refit is justified by the replay change and makes the selected inputs reviewable before healing.


In [17]:
# Save the bridge decision and index the versioned, matching-contract targets.
import hashlib
import json
from pathlib import Path

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def save_once(path, value):
    path = Path(path)
    if path.exists():
        if read_json(path) != value:
            raise RuntimeError('Saved decision artifact differs: ' + str(path))
    else:
        with path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(value, indent=2) + '\n')

student_path = Path('data/phase6/verified_student_replay/summary.json')
hidden_path = Path('data/phase6/aligned_hidden_capture/summary.json')
contract_path = Path('data/phase6/replay_contract.json')
inputs = {p.as_posix(): digest(p) for p in (student_path, hidden_path, contract_path)}
decision_path = Path('data/phase6/bridge_assessment.json')
index_path = Path('data/phase6/aligned_hidden_capture/target_index.json')
if decision_path.exists():
    decision = read_json(decision_path)
    if decision['input_sha256'] != inputs or digest(index_path) != decision['target_index_sha256']:
        raise RuntimeError('Decision inputs or target index changed.')
    print('Reusing the saved bridge decision; no metrics or selection recalculated.')
else:
    student, hidden = read_json(student_path), read_json(hidden_path)
    if student['status'] != 'passed' or hidden['status'] != 'passed':
        raise RuntimeError('Both evaluations must pass before the decision.')
    if student['max_reference_relative_l2'] > 0.001 or hidden['max_reference_relative_l2'] > 0.001:
        raise RuntimeError('Teacher control gate failed.')
    if student['optimizer_updates'] or hidden['optimizer_updates']:
        raise RuntimeError('Unexpected training during assessment.')
    handoff = read_json('data/phase6/handoff/manifest.json')
    capture_request = read_json('data/phase6/aligned_hidden_capture/request.json')
    artifacts = {a['name']: a for a in hidden['artifacts']}
    target_index = {'version': 'phase6-aligned-target-index-v1', 'capture_request_key': hidden['request_key'],
        'teacher_replay_contract_sha256': inputs[contract_path.as_posix()],
        'remote_directory': '/phase6/aligned_hidden_capture/' + hidden['request_key'], 'train': [], 'validation': []}
    for record in capture_request['request']['records']:
        name = record['capture_id'] + '.pt'
        artifact = artifacts[name]
        target_index['validation' if 'validation' in record['capture_id'] else 'train'].append({
            'capture_id': record['capture_id'], 'source_id': record['source_id'], 'dimension': record['dimension'],
            'request_key': hidden['request_key'], 'tensor_path': (hidden_path.parent / name).as_posix(),
            'tensor_sha256': artifact['sha256'], 'legacy_input_path': record['tensor_path'],
            'legacy_input_sha256': record['tensor_sha256'], 'records': 12})
    if len(target_index['train']) != 20 or len(target_index['validation']) != 10:
        raise RuntimeError('Aligned target index split counts differ.')
    save_once(index_path, target_index)
    global_rows = [r for r in student['aggregates'] if r['stage'] == 'all']
    fresh_rows = {r['layer']: r for r in hidden['bridge_hidden_aggregates'] if r['split'] == 'validation'}
    candidates = []
    for row in global_rows:
        recipe = next(c for c in handoff['candidates'] if c['candidate_id'] == row['candidate_id'])
        layer = recipe['removed_teacher_slots'][0]
        old = read_json('data/phase5/bridge_fitting_pilot/bridge_block_' + str(layer).zfill(2) + '_summary.json')
        fresh = fresh_rows[layer]
        candidates.append({'candidate_id': row['candidate_id'], 'layer': layer,
            'mean_velocity_relative_l2': row['trained_mean_relative_l2'],
            'velocity_error_reduction': row['relative_error_reduction'],
            'improved_velocity_comparisons': row['improved_comparisons'],
            'total_velocity_comparisons': row['comparisons'],
            'historical_validation_objective': old['selected_validation']['balanced_objective'],
            'fresh_validation_objective': fresh['trained']['balanced_objective'],
            'fresh_objective_relative_change': fresh['trained']['balanced_objective']/old['selected_validation']['balanced_objective'] - 1,
            'fresh_objective_reduction_from_identity': fresh['relative_objective_reduction']})
    supported = [r for r in candidates if r['velocity_error_reduction'] > 0 and r['fresh_objective_reduction_from_identity'] > 0]
    if not supported:
        raise RuntimeError('No existing bridge improves both local fit and complete velocity; assess refitting.')
    chosen = min(supported, key=lambda r: r['mean_velocity_relative_l2'])
    recipe = next(c for c in handoff['candidates'] if c['candidate_id'] == chosen['candidate_id'])
    decision = {'version': 'phase6-existing-bridge-assessment-v1', 'input_sha256': inputs,
        'target_index_path': index_path.as_posix(), 'target_index_sha256': digest(index_path),
        'decision': 'retain existing pretrained candidate for a healing pilot; no Phase 5 rerun justified solely by the replay change',
        'selected_candidate': recipe, 'candidate_evidence': candidates,
        'selection_policy': 'lowest equal-prompt/equal-stage mean validation velocity relative L2 among existing candidates improving both velocity and fresh local fit versus identity; no claim of statistical superiority',
        'original_targets_preserved': True, 'optimizer_updates': 0,
        'limitations': 'same small validation split used for Phase 5 checkpoint selection; no independent test, generated images, student trajectories, healing or speed benchmark',
        'teacher_contract': contract_path.as_posix()}
    save_once(decision_path, decision)
index = read_json(index_path)
for split in ('train', 'validation'):
    for record in index[split]:
        if digest(record['tensor_path']) != record['tensor_sha256']:
            raise RuntimeError('Aligned target artifact changed.')
if digest(decision['selected_candidate']['checkpoint_path']) != decision['selected_candidate']['checkpoint_sha256']:
    raise RuntimeError('Selected pretrained checkpoint changed.')
print('Decision:', decision['decision'])
print('Candidate:', decision['selected_candidate']['candidate_id'])
for row in decision['candidate_evidence']:
    print('Block', row['layer'], '| velocity relative L2:', row['mean_velocity_relative_l2'],
          '| historical hidden objective:', row['historical_validation_objective'],
          '| fresh hidden objective:', row['fresh_validation_objective'],
          '| objective relative change:', row['fresh_objective_relative_change'])
print('Aligned target index:', index_path, '| TRAIN:', len(index['train']), '| VALIDATION:', len(index['validation']))
print('Saved decision:', decision_path)


Reusing the saved bridge decision; no metrics or selection recalculated.
Decision: retain existing pretrained candidate for a healing pilot; no Phase 5 rerun justified solely by the replay change
Candidate: single_bridge_block_05
Block 2 | velocity relative L2: 0.12023494659612576 | historical hidden objective: 0.003969003023521509 | fresh hidden objective: 0.0039686774194706235 | objective relative change: -8.203673541085088e-05
Block 3 | velocity relative L2: 0.10238446481525898 | historical hidden objective: 0.005215348385390825 | fresh hidden objective: 0.005213093210477382 | objective relative change: -0.00043241117309822386
Block 4 | velocity relative L2: 0.05873384519169728 | historical hidden objective: 0.005036106774350628 | fresh hidden objective: 0.005036986034247092 | objective relative change: 0.00017459119432139403
Block 5 | velocity relative L2: 0.055178944828609626 | historical hidden objective: 0.006713706885348074 | fresh hidden objective: 0.006711873243330046 | objec

### Result and conclusion: retain the pretrained block-5 bridge for a healing pilot

The saved assessment selects **`single_bridge_block_05`**, using its original Phase 5 checkpoint from update **300**. It replaces original block 5 only, leaving **31 teacher Transformer blocks plus one rank-256 bridge** in the 32 original slots. Other candidates and every historical checkpoint remain preserved.

This candidate has the lowest average fixed-input validation velocity error among the four tested candidates: **5.518% relative L2**, versus **6.406%** for identity replacement of block 5. That is a **13.86% reduction** from identity, with improvements in **24 of 30 prompt-stage comparisons**. Its fresh hidden fitting objective improves by **61.40%** from identity, and differs from its historical fitting score by only **-0.027%**. Across all four bridges, the fresh-versus-historical objective changes are smaller than **0.05%** in magnitude. Together these results support keeping the existing pretrained weights; they do not justify a new Phase 5 fitting run solely because of the replay change.

The full selection recipe and evidence are saved in `data/phase6/bridge_assessment.json`. The separately versioned matching-contract teacher targets are indexed in `data/phase6/aligned_hidden_capture/target_index.json` (**20 TRAIN / 10 VALIDATION prompts**). This index records local paths, remote location, checksums, and provenance back to the unchanged original inputs. It provides refreshed bridge input/output targets for blocks 2?5 and the matching teacher velocities; it is not a capture of every surviving block.

This continuation completed **180 teacher forwards** across student validation and hidden capture, with **180 exact teacher reference matches**, plus **240 full student forwards** and frozen local bridge diagnostics. It performed **0 optimizer updates**. Each added result-producing cell has saved outputs and a plain-language explanation, and cache-only reruns reused the completed results without importing model or network libraries or repeating metrics and selection.

Block 5 is an operational choice for a small healing pilot, not a proven best architecture: block 4 is close, the same validation split was used in Phase 5, six block-5 comparisons regress against identity, and image quality, student trajectories, speed, and healing behavior remain unmeasured. The exact cause of the historical teacher mismatch also remains unresolved. Phase 6 architecture healing is still incomplete; no `phase5_v2.ipynb` is needed on the evidence collected here.


### What LoRA means, and what this healing pilot trains

**Bridge pretraining and healing are different training tasks.** Phase 5 trains only the rank-256 bridge to reproduce the hidden state after the removed teacher block. Phase 6 starts from that already-trained bridge and trains the modified student against the teacher's whole-model velocity prediction at the same prompt, noisy latent and timestep, adding hidden-state and bridge supervision. This velocity is the output at a sampled denoising step, not a finished image.

**LoRA means low-rank adaptation.** Instead of changing an existing large weight matrix `W`, we freeze it and add a small trainable correction: `y = W x + (alpha / rank) B A x`. Two small matrices `A` and `B` carry the correction. Our rank is **8** and alpha is **8**, giving scale **1**. This is the mechanism introduced in the [original LoRA paper](https://arxiv.org/abs/2106.09685); the concrete settings and counts here come from the saved pilot implementation.

For a **4,096 -> 4,096** attention projection, the original matrix has **16,777,216 weights**. LoRA adds a **4,096 -> 8 -> 4,096** linear path, with **65,536 trainable weights**, 256 times fewer than updating the whole matrix. It has **no GELU**. The rank-256 bridge is a separate nonlinear residual module replacing block 5; rank-8 LoRA modifies projections inside surviving blocks.

We insert LoRA into the query, key, value and output attention projections in all **31 surviving blocks**: **124 modules**, with **8,126,464 trainable LoRA parameters**. We also continue updating the bridge's **2,097,152 parameters**, giving **10,223,616 trainable parameters** in total. Original surviving weights remain frozen. Every LoRA up-projection starts at zero, making its correction zero, so the initial student exactly reproduces the Phase 5 pretrained-bridge baseline.

| Training task | Trainable parts | Main target |
|---|---|---|
| Phase 5 bridge pretraining | Bridge alone, 2,097,152 parameters | Hidden states after the removed block |
| Phase 6 pilot actually run | Pretrained bridge plus LoRA, 10,223,616 parameters | Whole-model velocity, with auxiliary hidden-state and bridge losses |
| Bridge-only healing, not run | Pretrained bridge alone | Whole-model velocity with the same auxiliary losses |

The purpose of LoRA here is to let surviving attention projections adjust to the imperfect replacement without optimizing every large original matrix. **Freezing the original weights does not freeze the block's behavior**: the LoRA correction can change its computed output. This is parameter-efficient end-to-end healing, not full-weight architecture healing.

Both saved 120-update pilots failed to improve validation. The before-healing student, already containing the pretrained bridge, has **5.518%** mean velocity error. The final checkpoints at learning rates 0.0001 and 0.00001 have **7.188%** and **6.113%**, respectively. Both runs select **update 0**. ?Healing? describes the intended recovery task; it does not guarantee recovery.

**This does not establish that LoRA caused the regression.** We have not run an otherwise-identical bridge-only Phase 6 output-distillation experiment. Phase 5 bridge pretraining uses a different target and optimization task, so it is not that ablation. The two failed recipes leave loss weighting, optimization and generalization unresolved. No training or metrics are recomputed by this explanation.

### Check differentiable prefix caches before healing

The first healing pilot trains the existing block-5 bridge and rank-8 attention adapters in all 31 surviving blocks; the large original weights remain frozen. FP32 optimizer parameters are cast to BF16 for inference-compatible projections. This tests end-to-end distillation without the memory cost of full-model Adam states. It is a parameter-efficient pilot, not full-weight architecture healing.

Before loading a GPU model, we check checkpointed prefix extraction and cached-stage backpropagation against a direct two-block computation. Extracted K/V are explicit checkpoint outputs, cached K/V are explicit inputs, and recomputation uses a private cache. A middle-stage loss must reproduce direct parameter/input gradients, reach the early prefix extraction, and leave shared caches untouched. We also check that a zero-initialized adapter exactly preserves its base projection. The checks are serialized and reused on rerun. They cover cache/gradient mechanics, not the real model's GPU behavior.


In [18]:
import hashlib
import json
from pathlib import Path

source_path = Path('data/phase6/healing_components.py')
source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
report_path = Path('data/phase6/healing_component_checks.json')
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    if report['source_sha256'] != source_sha or report['status'] != 'passed':
        raise RuntimeError('Component checks differ; preserve prior results.')
    print('Reusing saved component checks; no Torch import or gradient computation.')
else:
    import copy
    import torch
    from torch import nn
    from types import SimpleNamespace

    namespace = {}
    exec(compile(source_path.read_text(encoding='utf-8'), str(source_path), 'exec'), namespace)
    Wrapped = namespace['FunctionalCheckpointBlock']

    class Cache:
        def store(self, k, v):
            self.k, self.v = k, v
        def get(self):
            return self.k, self.v

    class ToyBlock(nn.Module):
        def __init__(self):
            super().__init__()
            self.projection = nn.Linear(4, 4, bias=False, dtype=torch.float64)
            self.attn = SimpleNamespace(processor=None)
        def forward(self, hidden_states, modulation, rotary_emb=None,
                    layer_cache=None, kv_cache_mode=None, cache_write_slice=None, **kwargs):
            projected = self.projection(hidden_states).tanh()
            if kv_cache_mode == 'extract':
                layer_cache.store(projected[:, cache_write_slice].clone(),
                                  (projected[:, cache_write_slice] * 0.7).clone())
            elif kv_cache_mode == 'cached':
                k, v = layer_cache.get()
                projected = projected + k.mean(1, keepdim=True) + v.mean(1, keepdim=True)
            return projected + modulation

    torch.manual_seed(20260929)
    direct = [ToyBlock(), ToyBlock()]
    wrapped = [Wrapped(copy.deepcopy(block), Cache) for block in direct]
    early = torch.randn(1, 5, 4, dtype=torch.float64)
    middle = torch.randn(1, 3, 4, dtype=torch.float64)
    modulation = torch.randn(1, 1, 4, dtype=torch.float64)

    def run(blocks):
        caches = [Cache(), Cache()]
        early_input = early.clone().requires_grad_()
        middle_input = middle.clone().requires_grad_()
        h = early_input
        for block, cache in zip(blocks, caches):
            h = block(h, modulation, layer_cache=cache, kv_cache_mode='extract',
                      cache_write_slice=slice(0, 2))
        # Like a middle-stage training sample, only the cached output has loss.
        prefix_objects = [cache.get() for cache in caches]
        h = middle_input
        for block, cache in zip(blocks, caches):
            h = block(h, modulation, layer_cache=cache, kv_cache_mode='cached')
        prediction = h.detach().clone()
        h.square().mean().backward()
        if any(cache.get()[0] is not saved[0] or cache.get()[1] is not saved[1]
               for cache, saved in zip(caches, prefix_objects)):
            raise RuntimeError('Backward recomputation mutated the shared prefix cache.')
        gradients = [p.grad.clone() for block in blocks for p in block.parameters()]
        return prediction, gradients, early_input.grad, middle_input.grad

    plain, checkpointed = run(direct), run(wrapped)
    maximum = 0.0
    for left, right in zip([plain[0], *plain[1], plain[2], plain[3]],
                           [checkpointed[0], *checkpointed[1], checkpointed[2], checkpointed[3]]):
        torch.testing.assert_close(left, right, atol=1e-12, rtol=1e-12)
        maximum = max(maximum, float((left-right).abs().max()))
    if not checkpointed[2].abs().sum() > 0:
        raise RuntimeError('Cached-stage gradients did not reach prefix extraction.')

    # Zero-initialized adapter must preserve the base linear's values exactly.
    base = nn.Linear(4, 4, bias=False, dtype=torch.float32)
    adapter = namespace['MasterLoRALinear'](base, rank=2, alpha=2)
    value = torch.randn(2, 3, 4)
    if not torch.equal(adapter(value), base(value)):
        raise RuntimeError('Zero adapter changed the baseline.')
    report = {'version': 'phase6-functional-cache-check-v1', 'status': 'passed',
              'source_sha256': source_sha, 'maximum_absolute_difference': maximum,
              'prefix_input_gradient_norm': float(checkpointed[2].norm()),
              'shared_cache_unchanged_after_backward': True,
              'zero_adapter_exact_baseline': True,
              'scope': 'two-block CPU float64 cache/gradient equivalence and zero-adapter identity; real-model GPU check still required'}
    with report_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(report, indent=2) + '\n')
print('Functional-cache checks:', report['status'])
print('Maximum direct/checkpoint difference:', report['maximum_absolute_difference'])
print('Gradient norm reaching prefix extraction:', report['prefix_input_gradient_norm'])
print('Shared cache preserved:', report['shared_cache_unchanged_after_backward'])
print('Zero adapter preserves baseline:', report['zero_adapter_exact_baseline'])


Functional-cache checks: passed
Maximum direct/checkpoint difference: 0.0
Gradient norm reaching prefix extraction: 0.4549865881281639
Shared cache preserved: True
Zero adapter preserves baseline: True


### Result: functional checkpointing preserves cached-stage gradients

The CPU two-block test produced **0.0 maximum difference** between direct and checkpointed predictions, parameter gradients, and input gradients. A middle-stage loss reached the early prefix input with gradient norm **0.45499**, while shared cache tensor objects remained unchanged after backward. The zero adapter also preserved its base projection exactly.

This validates the cache/checkpoint implementation on the small test. It does not establish real-model GPU behavior, memory use, or healing quality; the actual pilot must still pass a cached-stage forward/gradient preflight before any optimizer update. Results are saved in `data/phase6/healing_component_checks.json`.


### Run the block-5 parameter-efficient healing pilot

We initialize the selected block-5 bridge from its preserved Phase 5 checkpoint, then add zero-initialized rank-8 adapters to the query, key, value, and output projections of all 31 surviving attention blocks. We train **10,223,616 FP32 master parameters** (bridge plus 124 adapters); the original surviving weights remain frozen, and projections use BF16. This is end-to-end distillation of a pruned student with adapters, not full-weight fine-tuning.

A fresh audited teacher first checks all 90 saved reference velocities. The initial adapted student must then reproduce the saved unhealed block-5 student's 30 validation outputs within the unchanged 0.001 relative-L2 threshold. Before any optimizer update, a real TRAIN middle-stage gradient check compares checkpointed and inference outputs, requires finite bridge/adapter and prefix-cache gradients, verifies unchanged shared cache objects after backward, and verifies that frozen base weights receive no gradients. Any failure is saved and stops that immutable attempt.

The optimizer performs **120 updates**: two shuffled passes over the 20 TRAIN prompts times three saved stages. Each update uses a fresh cache; middle and late samples rebuild the prefix with current student parameters under gradient tracking. The loss is normalized velocity MSE (squared relative L2), plus 0.1 times normalized-vector hidden distillation at surviving slots 2/3/4 and 0.1 times supervision of the bridge output at slot 5. Hidden losses weight available prefix/image roles equally. AdamW uses learning rate 0.0001, weight decay 0.01, and gradient clipping at 1.0. No validation gradients are used.

We evaluate at updates 0, 30, 60, 90, and 120 and select the lowest equally weighted validation mean velocity relative L2, retaining update 0 if training never improves it. We preserve the sampling schedule, initial state, real gradient-check outputs, every evaluation, and model/optimizer/RNG checkpoints every 30 updates. Handled failures also save emergency optimizer state. The selected checkpoint is reloaded and checked against its saved predictions, and every surviving frozen parameter is audited. The controller downloads all saved artifacts, including handled failures; reruns reuse local or durable remote results without repeating model calls or optimization. Image quality, student trajectories, independent test performance, and speed are outside this pilot.


In [19]:
# Compare independent BF16 students on cached validation inputs, with durable reuse.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
contract = read_json('data/phase6/replay_contract.json')
if contract['status'] != 'verified_for_saved_pilot' or contract['verified_controls'] != 90:
    raise RuntimeError('A verified replay contract is required.')
assessment = read_json('data/phase6/bridge_assessment.json')
for name, expected in assessment['input_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Assessment source changed: ' + name)
if digest(assessment['target_index_path']) != assessment['target_index_sha256']:
    raise RuntimeError('Aligned target index changed.')
aligned_index = read_json(assessment['target_index_path'])
for split in ('train', 'validation'):
    for record in aligned_index[split]:
        if digest(record['tensor_path']) != record['tensor_sha256']:
            raise RuntimeError('Aligned target changed.')
components_path = Path('data/phase6/healing_components.py')
component_checks = read_json('data/phase6/healing_component_checks.json')
if component_checks['status'] != 'passed' or component_checks['source_sha256'] != digest(components_path):
    raise RuntimeError('Functional cache component checks are required.')
worker_path = Path('data/phase6/healing_pilot_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-parameter-efficient-healing-pilot-v1', 'mode': 'verify',
    'teacher': handoff['teacher'],
    'handoff_manifest_sha256': digest(handoff_path),
    'replay_contract_sha256': digest('data/phase6/replay_contract.json'),
    'assessment_sha256': digest('data/phase6/bridge_assessment.json'),
    'component_checks_sha256': digest('data/phase6/healing_component_checks.json'),
    'expected_state': contract['expected_model_state'],
    'arithmetic_contract': contract['arithmetic_contract'],
    'runtime_environment': contract['runtime_environment'],
    'reference_key': contract['generation_request_key'],
    'reference_report_semantic_sha256': contract['generation_report_semantic_sha256'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'selected_candidate': assessment['selected_candidate'],
    'fit_key': handoff['phase5_fitting_request_key'],
    'aligned_target_index': aligned_index,
    'previous_replay_report': read_json('data/phase6/verified_student_replay/summary.json'),
    'wrapper_source': Path('data/phase4/bridge_block_definition.py').read_text(encoding='utf-8'),
    'components_source': components_path.read_text(encoding='utf-8'),
    'components_sha256': digest(components_path), 'worker_source': worker_source,
    'adapter_rank': 8, 'adapter_alpha': 8, 'seed': 20260929, 'sampling_seed': 20260930,
    'updates': 120, 'checkpoint_every': 30, 'learning_rate': 0.0001,
    'weight_decay': 0.01, 'gradient_clip': 1.0,
    'hidden_weight': 0.1, 'bridge_weight': 0.1,
    'optimizer': 'AdamW on FP32 bridge and adapter masters; original surviving weights frozen',
    'sampling': 'two independently shuffled passes over all 20 TRAIN prompts and their three stages; batch one',
    'objective': 'relative velocity MSE + 0.1 mean normalized-vector squared distance at surviving slots 2/3/4 + 0.1 corresponding bridge-slot-5 distance; equal available prefix/image role weights',
    'cache_policy': 'fresh student cache each update/evaluation prompt; middle/late rebuild differentiable prefix from saved early input; functional checkpointing snapshots cached K/V and returns extracted K/V',
    'selection': 'minimum equal-prompt/equal-stage VALIDATION mean velocity relative L2 at step 0/30/60/90/120; preserve step 0 if no improvement',
    'checkpoint_policy': 'save baseline, gradient smoke, schedule, every 30-update optimizer/model/RNG checkpoint and validation outputs, then reload and verify selected checkpoint; save emergency state on handled failure',
    'scope': 'parameter-efficient healing implementation pilot on saved TRAIN/VALIDATION; no full-weight fine-tuning, generated images or independent test',
}

key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/healing_pilot')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/healing_pilot/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'passed':
        raise RuntimeError('Healing pilot failed; saved state preserved, no automatic retry: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 90 or report['student_calls'] != 384 or report['optimizer_updates'] != 120:
        raise RuntimeError('Unexpected pilot forward/update counts.')
    if report['max_reference_relative_l2'] > 0.001 or not report['frozen_parameters_unchanged']:
        raise RuntimeError('Teacher control or frozen-weight audit failed.')
    if report['validation_optimizer_updates'] != 0 or len(report['evaluations']) != 5:
        raise RuntimeError('Validation/checkpoint policy differs.')

complete = False
if report is not None:
    if report['request_key'] != key:
        raise RuntimeError('Report provenance differs.')
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        try:
            json.loads(b''.join(results_volume.read_file(remote_dir + '/started.json')))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('This GPU attempt already started; inspect its progress and wait for its durable report. No GPU is rescheduled.')
        app = modal.App('nihonga-phase6-healing-pilot')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=7200, cpu=8, memory=65536,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['heal_student'])
        with modal.enable_output(), app.run(detach=True):
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
validate(report)
print('Healing pilot:', report['status'], '| GPU:', report['gpu'])
print('Teacher forwards:', report['teacher_calls'], '| student forwards:', report['student_calls'], '| optimizer updates:', report['optimizer_updates'])
print('Trainable parameters:', report['trainable_parameters'], '| adapter modules:', len(report['adapter_modules']))
print('Gradient preflight:', report['gradient_smoke']['inference_equivalence'])
for evaluation in report['evaluations']:
    print('Step', evaluation['step'], '| validation mean relative L2:', evaluation['mean_relative_l2'], '| maximum:', evaluation['max_relative_l2'])
print('Selected step:', report['selected_step'], '| error reduction:', report['selected_relative_error_reduction'])
print('Improved comparisons:', report['selected_improved_comparisons'], '/30')
print('Frozen parameters checked:', report['frozen_parameter_audit_count'])
print('Selected checkpoint:', run_dir / report['selected_checkpoint'])
print('Saved pilot artifacts:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Healing pilot: passed | GPU: NVIDIA H100 80GB HBM3
Teacher forwards: 90 | student forwards: 384 | optimizer updates: 120
Trainable parameters: 10223616 | adapter modules: 124
Gradient preflight: {'relative_l2_error': 0.0, 'max_abs_error': 0.0, 'exact_equal': True}
Step 0 | validation mean relative L2: 0.055178944828609626 | maximum: 0.15094755589962006
Step 30 | validation mean relative L2: 0.08265145439654589 | maximum: 0.2819667160511017
Step 60 | validation mean relative L2: 0.08781036846339703 | maximum: 0.2376486361026764
Step 90 | validation mean relative L2: 0.10283102442820867 | maximum: 0.4410056471824646
Step 120 | validation mean relative L2: 0.07188390928010145 | maximum: 0.18164262175559998
Selected step: 0 | error reduction: 0.0
Improved comparisons: 0 /30
Frozen parameters checked: 288
Selected checkpoint: data\phase6\healing_pilot\checkpoint_step_000.pt
Saved pilot artifacts: data\phase6\heali

### Result: the first learning rate worsens validation; keep the unhealed baseline

The pilot completed **120 optimizer updates**, but **every trained checkpoint had higher validation error than update 0**. The best selected checkpoint is therefore **update 0**, not a trained checkpoint. The report's `passed` status means the execution, gradient, provenance, and reload checks passed; it does not mean healing improved performance.

| Update | Validation mean velocity error | Change from baseline (percentage points) |
| --- | --- | --- |
| 0 | 5.518% | +0.000 |
| 30 | 8.265% | +2.747 |
| 60 | 8.781% | +3.263 |
| 90 | 10.283% | +4.765 |
| 120 | 7.188% | +1.670 |

The second column measures relative velocity error against the teacher. The third column is an **absolute difference in percentage points**, not a relative percentage reduction. For example, update 120 increases error from 5.518% to 7.188%, a rise of about 1.670 percentage points. We retain the baseline and all unsuccessful trained states rather than promote the last checkpoint.

All **90 teacher controls** and all **30 initial student baseline controls** matched exactly. The real cached-stage gradient forward was also exactly equal to its inference counterpart; finite nonzero gradients reached both trainable groups and K/V prefixes at tested slots 0, 4, 6, and 31. Backward left shared caches untouched. All **288 surviving frozen parameter tensors** remained byte-identical to the pinned teacher. Reloading the selected baseline reproduced all **30 validation predictions exactly**. These controls support the implementation's correctness, while the validation result shows that this training configuration does not help this pilot.

The run used **10,223,616 trainable parameters**, **384 full student forwards**, and **0 validation optimizer updates**. Peak allocated training tensor memory was **24.79 GiB**; this is PyTorch allocated memory, not total device or reserved memory. All 43 tensor artifacts, including optimizer/RNG checkpoints, schedule, gradient diagnostics, and predictions, are saved locally and remotely. A local cached rerun made no model/network imports or optimization calls and recalculated no metrics.

The validation regression does not establish its cause. The learning rate of 0.0001 is one plausible contributor, so a controlled lower-rate run can test that explanation while preserving this result. No image-quality or student-trajectory conclusion is supported here.


### Test a ten-times smaller healing learning rate

The first pilot did not improve validation. We repeat the same 120-update protocol with learning rate **0.00001**, ten times smaller, retaining the same model, pretrained bridge, adapter rank and initialization seed, sampling seed and schedule, loss weights, data, numerical settings, checkpoint spacing, and validation selection rule. This is a paired learning-rate experiment, not a continuation of the unsuccessful trained weights.

The fresh process must repeat the teacher and initial student controls and the real cached-stage gradient preflight before training. Its immutable request, outputs, and checkpoints are saved separately under `data/phase6/healing_pilot_lr_low`; the first run remains preserved. Selecting among both runs uses the same small validation split and increases selection bias, so this remains an implementation pilot rather than an independent generalization result. The unchanged baseline remains eligible if this rate also fails to improve validation.


In [20]:
# Compare independent BF16 students on cached validation inputs, with durable reuse.
import hashlib
import json
from pathlib import Path

root = Path.cwd()
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

handoff_path = Path('data/phase6/handoff/manifest.json')
handoff_summary = read_json('data/phase6/handoff/summary.json')
if digest(handoff_path) != handoff_summary['manifest_sha256']:
    raise RuntimeError('Handoff manifest changed.')
handoff = read_json(handoff_path)
for name, expected in handoff['artifact_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Handoff artifact changed: ' + name)
contract = read_json('data/phase6/replay_contract.json')
if contract['status'] != 'verified_for_saved_pilot' or contract['verified_controls'] != 90:
    raise RuntimeError('A verified replay contract is required.')
assessment = read_json('data/phase6/bridge_assessment.json')
for name, expected in assessment['input_sha256'].items():
    if digest(name) != expected:
        raise RuntimeError('Assessment source changed: ' + name)
if digest(assessment['target_index_path']) != assessment['target_index_sha256']:
    raise RuntimeError('Aligned target index changed.')
aligned_index = read_json(assessment['target_index_path'])
for split in ('train', 'validation'):
    for record in aligned_index[split]:
        if digest(record['tensor_path']) != record['tensor_sha256']:
            raise RuntimeError('Aligned target changed.')
components_path = Path('data/phase6/healing_components.py')
component_checks = read_json('data/phase6/healing_component_checks.json')
if component_checks['status'] != 'passed' or component_checks['source_sha256'] != digest(components_path):
    raise RuntimeError('Functional cache component checks are required.')
worker_path = Path('data/phase6/healing_pilot_worker.py')
worker_source = worker_path.read_text(encoding='utf-8')
request = {
    'version': 'phase6-parameter-efficient-healing-pilot-low-lr-v2', 'mode': 'verify',
    'teacher': handoff['teacher'],
    'handoff_manifest_sha256': digest(handoff_path),
    'replay_contract_sha256': digest('data/phase6/replay_contract.json'),
    'assessment_sha256': digest('data/phase6/bridge_assessment.json'),
    'component_checks_sha256': digest('data/phase6/healing_component_checks.json'),
    'expected_state': contract['expected_model_state'],
    'arithmetic_contract': contract['arithmetic_contract'],
    'runtime_environment': contract['runtime_environment'],
    'reference_key': contract['generation_request_key'],
    'reference_report_semantic_sha256': contract['generation_report_semantic_sha256'],
    'records': handoff['target_index']['train'] + handoff['target_index']['validation'],
    'transformer_source_sha256': handoff['artifact_sha256']['data/phase4/transformer_qwenimage21_pinned.py'],
    'selected_candidate': assessment['selected_candidate'],
    'fit_key': handoff['phase5_fitting_request_key'],
    'aligned_target_index': aligned_index,
    'previous_replay_report': read_json('data/phase6/verified_student_replay/summary.json'),
    'wrapper_source': Path('data/phase4/bridge_block_definition.py').read_text(encoding='utf-8'),
    'components_source': components_path.read_text(encoding='utf-8'),
    'components_sha256': digest(components_path), 'worker_source': worker_source,
    'adapter_rank': 8, 'adapter_alpha': 8, 'seed': 20260929, 'sampling_seed': 20260930,
    'updates': 120, 'checkpoint_every': 30, 'learning_rate': 0.00001,
    'weight_decay': 0.01, 'gradient_clip': 1.0,
    'hidden_weight': 0.1, 'bridge_weight': 0.1,
    'optimizer': 'AdamW on FP32 bridge and adapter masters; original surviving weights frozen',
    'sampling': 'two independently shuffled passes over all 20 TRAIN prompts and their three stages; batch one',
    'objective': 'relative velocity MSE + 0.1 mean normalized-vector squared distance at surviving slots 2/3/4 + 0.1 corresponding bridge-slot-5 distance; equal available prefix/image role weights',
    'cache_policy': 'fresh student cache each update/evaluation prompt; middle/late rebuild differentiable prefix from saved early input; functional checkpointing snapshots cached K/V and returns extracted K/V',
    'selection': 'minimum equal-prompt/equal-stage VALIDATION mean velocity relative L2 at step 0/30/60/90/120; preserve step 0 if no improvement',
    'checkpoint_policy': 'save baseline, gradient smoke, schedule, every 30-update optimizer/model/RNG checkpoint and validation outputs, then reload and verify selected checkpoint; save emergency state on handled failure',
    'scope': 'parameter-efficient healing implementation pilot on saved TRAIN/VALIDATION; no full-weight fine-tuning, generated images or independent test',
}

key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
run_dir = Path('data/phase6/healing_pilot_lr_low')
run_dir.mkdir(exist_ok=True)
manifest_path = run_dir / 'request.json'
manifest = {'request': request, 'request_key': key}
if manifest_path.exists():
    if read_json(manifest_path) != manifest:
        raise RuntimeError('Replay request changed; preserve saved results.')
else:
    with manifest_path.open('x', encoding='utf-8') as handle:
        handle.write(json.dumps(manifest, indent=2) + '\n')
report_path = run_dir / 'summary.json'
remote_dir = f'/phase6/healing_pilot/{key}'
report = read_json(report_path) if report_path.exists() else None

def validate(report):
    if report['request_key'] != key or report['status'] != 'passed':
        raise RuntimeError('Healing pilot failed; saved state preserved, no automatic retry: ' + report.get('error', 'unknown'))
    if report['teacher_calls'] != 90 or report['student_calls'] != 384 or report['optimizer_updates'] != 120:
        raise RuntimeError('Unexpected pilot forward/update counts.')
    if report['max_reference_relative_l2'] > 0.001 or not report['frozen_parameters_unchanged']:
        raise RuntimeError('Teacher control or frozen-weight audit failed.')
    if report['validation_optimizer_updates'] != 0 or len(report['evaluations']) != 5:
        raise RuntimeError('Validation/checkpoint policy differs.')

complete = False
if report is not None:
    if report['request_key'] != key:
        raise RuntimeError('Report provenance differs.')
    complete = all((run_dir / a['name']).exists() for a in report['artifacts'])
    for a in report['artifacts']:
        path = run_dir / a['name']
        if path.exists() and digest(path) != a['sha256']:
            raise RuntimeError('Saved replay tensor changed.')
if complete:
    print('Reusing verified replay results; no model imports, network calls or forwards.')
else:
    import modal
    runtime = handoff['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    try:
        remote_report = json.loads(b''.join(results_volume.read_file(remote_dir + '/summary.json')))
        print('Recovering durable remote replay; no GPU scheduled.')
    except FileNotFoundError:
        if report is not None or any(run_dir.glob('*.pt')):
            raise RuntimeError('Local results are incomplete and no remote report exists; inspect before retrying.')
        try:
            json.loads(b''.join(results_volume.read_file(remote_dir + '/started.json')))
        except FileNotFoundError:
            pass
        else:
            raise RuntimeError('This GPU attempt already started; inspect its progress and wait for its durable report. No GPU is rescheduled.')
        app = modal.App('nihonga-phase6-healing-pilot-low-lr')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
            .pip_install(*runtime['packages'])
            .env(request['runtime_environment']))
        weights_volume = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(worker_source, str(worker_path), 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=7200, cpu=8, memory=65536,
            max_containers=1, retries=0,
            volumes={'/phase3': results_volume, '/root/.cache/huggingface': weights_volume.with_mount_options(read_only=True)})(namespace['heal_student'])
        with modal.enable_output(), app.run(detach=True):
            remote_report = worker.remote(request, key)
    if remote_report['request_key'] != key:
        raise RuntimeError('Remote report provenance differs.')
    if report is not None and report != remote_report:
        raise RuntimeError('Local and remote replay reports differ.')
    if not report_path.exists():
        with report_path.open('x', encoding='utf-8') as handle:
            handle.write(json.dumps(remote_report, indent=2) + '\n')
    report = remote_report
    for artifact in report['artifacts']:
        path = run_dir / artifact['name']
        if path.exists():
            if digest(path) != artifact['sha256']:
                raise RuntimeError('Local velocity artifact differs.')
            continue
        temporary = path.with_suffix('.download')
        with temporary.open('wb') as handle:
            for chunk in results_volume.read_file(remote_dir + '/' + artifact['name']):
                handle.write(chunk)
        if digest(temporary) != artifact['sha256']:
            raise RuntimeError('Downloaded velocities differ.')
        temporary.replace(path)
validate(report)
print('Healing pilot:', report['status'], '| GPU:', report['gpu'])
print('Teacher forwards:', report['teacher_calls'], '| student forwards:', report['student_calls'], '| optimizer updates:', report['optimizer_updates'])
print('Trainable parameters:', report['trainable_parameters'], '| adapter modules:', len(report['adapter_modules']))
print('Gradient preflight:', report['gradient_smoke']['inference_equivalence'])
for evaluation in report['evaluations']:
    print('Step', evaluation['step'], '| validation mean relative L2:', evaluation['mean_relative_l2'], '| maximum:', evaluation['max_relative_l2'])
print('Selected step:', report['selected_step'], '| error reduction:', report['selected_relative_error_reduction'])
print('Improved comparisons:', report['selected_improved_comparisons'], '/30')
print('Frozen parameters checked:', report['frozen_parameter_audit_count'])
print('Selected checkpoint:', run_dir / report['selected_checkpoint'])
print('Saved pilot artifacts:', run_dir)


Reusing verified replay results; no model imports, network calls or forwards.
Healing pilot: passed | GPU: NVIDIA H100 80GB HBM3
Teacher forwards: 90 | student forwards: 384 | optimizer updates: 120
Trainable parameters: 10223616 | adapter modules: 124
Gradient preflight: {'relative_l2_error': 0.0, 'max_abs_error': 0.0, 'exact_equal': True}
Step 0 | validation mean relative L2: 0.055178944828609626 | maximum: 0.15094755589962006
Step 30 | validation mean relative L2: 0.06041828797509273 | maximum: 0.16981394588947296
Step 60 | validation mean relative L2: 0.06185537201041977 | maximum: 0.16065560281276703
Step 90 | validation mean relative L2: 0.06106605188300212 | maximum: 0.17243501543998718
Step 120 | validation mean relative L2: 0.06113102845847607 | maximum: 0.18038393557071686
Selected step: 0 | error reduction: 0.0
Improved comparisons: 0 /30
Frozen parameters checked: 288
Selected checkpoint: data\phase6\healing_pilot_lr_low\checkpoint_step_000.pt
Saved pilot artifacts: data\ph

### Result: the smaller learning rate still does not improve validation

This second run completed all **120 optimizer updates**, with all 43 reported tensor artifacts saved locally. Teacher replay, baseline reproduction, cached-stage gradient checks, selected-checkpoint reload and the 288-tensor frozen-base audit passed. The notebook cell reused the saved run and checked its artifacts without importing Torch, Modal or Diffusers or making any model calls.

| Update | Mean validation velocity error |
|---:|---:|
| 0 | 5.518% |
| 30 | 6.042% |
| 60 | 6.186% |
| 90 | 6.107% |
| 120 | 6.113% |

The baseline is **5.518%**. The ten-times smaller learning rate ends at **6.113%**, approximately **0.595 percentage points worse** than the baseline. It causes less degradation than the first run's final 7.188%, but no evaluated trained checkpoint beats the initial pretrained bridge. Both runs therefore select **update 0**, with **0% healing error reduction**. A passed execution status records correct mechanics and preserved artifacts; it does not indicate a successful improvement.

We keep the Phase 5 block-5 checkpoint. These two rates on 20 training prompts do not establish why healing regresses: loss weighting, optimization and generalization remain uncertain. The same 10 validation prompts have been reused for selection, so this is still a development pilot. Full-weight healing, generated-image quality and student inference efficiency remain unmeasured. All negative checkpoints and optimizer states remain available for investigation.
